In [0]:
%pip install node2vec networkx gensim scikit-learn matplotlib seaborn umap-learn hdbscan faiss-cpu -q

# 📊 Graph Algorithms: DeepWalk & Node2Vec — A Comprehensive Study

## Introduction to Graph Representation Learning

Graphs are ubiquitous data structures representing relationships between entities — social networks, biological networks, knowledge graphs, transaction networks, and more. However, most machine learning algorithms require **fixed-size numeric feature vectors** as input, not graph structures.

**Graph Representation Learning** (also called **Network Embedding**) solves this by learning a mapping:

$$f: V \rightarrow \mathbb{R}^d$$

where each node $v \in V$ is mapped to a $d$-dimensional dense vector that preserves the structural properties of the graph.

### Why Not Just Use Adjacency Matrices?

| Property | Adjacency Matrix | Node Embeddings |
|----------|-----------------|----------------|
| Dimensionality | $O(|V|)$ — grows with graph size | Fixed $d$ (e.g., 64 or 128) |
| Sparsity | Extremely sparse for real graphs | Dense vectors |
| ML compatibility | Poor (high-dim, sparse) | Excellent (low-dim, dense) |
| Captures higher-order structure | No (only direct connections) | Yes (via random walks) |

### The Key Insight

Both **DeepWalk** and **Node2Vec** are inspired by **Word2Vec** from NLP. The analogy:

| NLP (Word2Vec) | Graphs (DeepWalk/Node2Vec) |
|---------------|---------------------------|
| Corpus of text | Graph |
| Sentence | Random walk |
| Word | Node |
| Word embedding | Node embedding |

Just as words appearing in similar contexts have similar meanings (distributional hypothesis), **nodes visited in similar random walk contexts have similar structural roles**.

## 0.1 DeepWalk

### Paper
> Perozzi, B., Al-Rfou, R., & Skiena, S. (2014). *DeepWalk: Online Learning of Social Representations.* KDD 2014.

### Core Idea

DeepWalk learns node embeddings by:
1. **Generating random walks** from each node (simulating "sentences" on the graph)
2. **Applying Skip-gram (Word2Vec)** on the generated walks to learn embeddings

### Algorithm

```
Algorithm: DeepWalk(G, w, d, γ, t)
Input:
  G = (V, E)  — graph
  w           — window size
  d           — embedding dimension
  γ (gamma)   — number of walks per node
  t           — walk length

Output:
  Φ ∈ ℝ^(|V| × d)  — matrix of node embeddings

1. Initialize Φ with random values
2. Build a binary tree T from V (for Hierarchical Softmax)
3. for i = 1 to γ do:
4.     O = Shuffle(V)          // Random ordering of nodes
5.     for each v_i ∈ O do:
6.         W_vi = RandomWalk(G, v_i, t)  // Generate walk of length t
7.         SkipGram(Φ, W_vi, w)          // Update embeddings
8. return Φ
```

### Mathematical Formulation

#### Random Walk Generation

A random walk of length $t$ starting at node $v_0$ is a stochastic process:

$$W = (v_0, v_1, v_2, \ldots, v_t)$$

where the transition probability is:

$$P(v_{i+1} = x \mid v_i = v) = \begin{cases} \frac{1}{|\mathcal{N}(v)|} & \text{if } (v, x) \in E \\ 0 & \text{otherwise} \end{cases}$$

where $\mathcal{N}(v)$ is the set of neighbors of $v$. This is a **uniform random walk** — each neighbor is equally likely.

#### Skip-gram Objective

Given a walk $W = (v_0, v_1, \ldots, v_t)$, the Skip-gram objective maximizes:

$$\max_\Phi \sum_{v_i \in W} \sum_{-w \leq j \leq w, j \neq 0} \log P(v_{i+j} \mid \Phi(v_i))$$

where $w$ is the context window size and $\Phi(v_i) \in \mathbb{R}^d$ is the embedding of node $v_i$.

The probability is modeled using **softmax**:

$$P(v_j \mid v_i) = \frac{\exp(\Phi(v_j)^\top \Phi(v_i))}{\sum_{v_k \in V} \exp(\Phi(v_k)^\top \Phi(v_i))}$$

Since computing the full softmax is expensive ($O(|V|)$), DeepWalk uses **Hierarchical Softmax** with a binary tree, reducing complexity to $O(\log|V|)$.

#### Hierarchical Softmax

Each node $v_k$ is assigned a path in a binary tree from root to leaf. If the path to node $v_k$ has length $L$ and $b_l \in \{-1, +1\}$ indicates left/right at level $l$:

$$P(v_k \mid \Phi(v_i)) = \prod_{l=1}^{L} \sigma(b_l \cdot \Phi(v_i)^\top \psi_l)$$

where $\sigma(x) = \frac{1}{1+e^{-x}}$ is the sigmoid function and $\psi_l$ are parameters of the internal tree nodes.

### Key Hyperparameters

| Parameter | Symbol | Typical Values | Effect |
|-----------|--------|---------------|--------|
| Walk length | $t$ | 40–80 | Longer walks capture broader context |
| Walks per node | $\gamma$ | 10–80 | More walks = better coverage |
| Window size | $w$ | 5–10 | Larger window = more global structure |
| Embedding dim | $d$ | 64–256 | Higher = more expressive but slower |

In [0]:
import numpy as np
import networkx as nx
from gensim.models import Word2Vec
import random
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# DeepWalk Implementation from Scratch
# ============================================================

class DeepWalk:
    """
    DeepWalk: Online Learning of Social Representations
    
    Learns node embeddings by:
    1. Generating uniform random walks from each node
    2. Applying Word2Vec (Skip-gram) on the walks
    """
    
    def __init__(self, graph, walk_length=40, num_walks=10, 
                 embedding_dim=64, window_size=5, workers=4, seed=42):
        """
        Parameters:
        -----------
        graph : networkx.Graph
            Input graph (undirected)
        walk_length : int
            Length of each random walk (t)
        num_walks : int  
            Number of walks per node (γ)
        embedding_dim : int
            Dimensionality of embeddings (d)
        window_size : int
            Context window size for Skip-gram (w)
        workers : int
            Number of parallel workers for Word2Vec
        seed : int
            Random seed for reproducibility
        """
        self.graph = graph
        self.walk_length = walk_length
        self.num_walks = num_walks
        self.embedding_dim = embedding_dim
        self.window_size = window_size
        self.workers = workers
        self.seed = seed
        self.model = None
        
    def _random_walk(self, start_node):
        """
        Generate a single uniform random walk starting from start_node.
        
        P(next = x | current = v) = 1/|N(v)| if (v,x) ∈ E, else 0
        """
        walk = [start_node]
        current = start_node
        
        for _ in range(self.walk_length - 1):
            neighbors = list(self.graph.neighbors(current))
            if len(neighbors) == 0:
                break  # Dead end — stop walk
            current = random.choice(neighbors)  # Uniform selection
            walk.append(current)
            
        return walk
    
    def _generate_walks(self):
        """
        Generate γ random walks for each node in the graph.
        Nodes are shuffled before each iteration (as per paper).
        """
        walks = []
        nodes = list(self.graph.nodes())
        
        for _ in range(self.num_walks):
            random.shuffle(nodes)  # Shuffle nodes each iteration
            for node in nodes:
                walk = self._random_walk(node)
                # Convert to strings for Word2Vec
                walks.append([str(n) for n in walk])
                
        return walks
    
    def fit(self):
        """
        Train the DeepWalk model:
        1. Generate random walks
        2. Train Word2Vec Skip-gram on walks
        """
        random.seed(self.seed)
        
        print(f"Generating {self.num_walks} walks per node "
              f"(total: {self.num_walks * self.graph.number_of_nodes()} walks)...")
        walks = self._generate_walks()
        
        print(f"Training Word2Vec Skip-gram (dim={self.embedding_dim}, "
              f"window={self.window_size})...")
        self.model = Word2Vec(
            sentences=walks,
            vector_size=self.embedding_dim,
            window=self.window_size,
            min_count=0,  # Include all nodes
            sg=1,  # Skip-gram (NOT CBOW)
            hs=1,  # Hierarchical softmax (as per paper)
            workers=self.workers,
            seed=self.seed,
            epochs=5
        )
        
        print(f"DeepWalk training complete! "
              f"Learned embeddings for {len(self.model.wv)} nodes.")
        return self
    
    def get_embedding(self, node):
        """Get embedding vector for a specific node."""
        return self.model.wv[str(node)]
    
    def get_all_embeddings(self):
        """Get embedding matrix for all nodes (sorted by node id)."""
        nodes = sorted(self.graph.nodes())
        embeddings = np.array([self.model.wv[str(n)] for n in nodes])
        return nodes, embeddings
    
    def most_similar(self, node, topn=10):
        """Find most similar nodes in embedding space."""
        return self.model.wv.most_similar(str(node), topn=topn)


print("✅ DeepWalk class defined successfully")

## 0.2 Node2Vec

### Paper
> Grover, A., & Leskovec, J. (2016). *node2vec: Scalable Feature Learning for Networks.* KDD 2016.

### Key Innovation over DeepWalk

DeepWalk uses **uniform random walks**. Node2Vec introduces **biased random walks** controlled by two parameters:
- **$p$ (Return parameter)**: Controls likelihood of returning to the previous node
- **$q$ (In-out parameter)**: Controls whether the walk explores outward (BFS-like) or stays local (DFS-like)

This allows Node2Vec to interpolate between:
- **BFS (Breadth-First Search)** behavior → captures **local/structural** equivalence
- **DFS (Depth-First Search)** behavior → captures **global/homophily** (community) patterns

### Biased Random Walk — The $\alpha$ Function

Given a walk that just traversed edge $(t, v)$ and is now at node $v$, the unnormalized transition probability to node $x$ is:

$$\alpha_{pq}(t, x) = \begin{cases} \frac{1}{p} & \text{if } d_{tx} = 0 \quad \text{(return to } t\text{)} \\ 1 & \text{if } d_{tx} = 1 \quad \text{(}x\text{ is also neighbor of } t\text{)} \\ \frac{1}{q} & \text{if } d_{tx} = 2 \quad \text{(}x\text{ moves away from } t\text{)} \end{cases}$$

where $d_{tx}$ is the shortest-path distance between $t$ and $x$ (always 0, 1, or 2 for neighbors of $v$).

The normalized transition probability is:

$$P(v_{i+1} = x \mid v_i = v, v_{i-1} = t) = \frac{\alpha_{pq}(t, x) \cdot w_{vx}}{Z}$$

where $w_{vx}$ is the edge weight and $Z = \sum_{y \in \mathcal{N}(v)} \alpha_{pq}(t, y) \cdot w_{vy}$ is the normalization constant.

### Intuition Behind $p$ and $q$

#### Return parameter $p$:
- **Low $p$** (e.g., 0.25): High probability of returning → walk stays very local, revisits nodes
- **High $p$** (e.g., 4): Low probability of returning → walk moves outward, explores

#### In-out parameter $q$:
- **Low $q$** (e.g., 0.5): High probability of moving away → **DFS-like**, explores far from source, captures **homophily** (nodes in same community get similar embeddings)
- **High $q$** (e.g., 2): Low probability of moving away → **BFS-like**, stays near source, captures **structural equivalence** (structurally similar nodes get similar embeddings even if far apart)

### Visual Intuition

```
              t ←——— v ———→ x₁  (d_tx₁ = 2, weight = 1/q)
              ↑      |      
              |      ↓
              t ←——— v ———→ x₂  (d_tx₂ = 1, weight = 1)
              ↑             ↑
              └—————————————┘    (x₂ is neighbor of both v and t)
              
              t ←——— v ———→ t   (d_tt = 0, weight = 1/p)
```

### Optimization Objective

Same as DeepWalk — maximize log-likelihood of observing network neighborhoods:

$$\max_f \sum_{u \in V} \left[ -\log Z_u + \sum_{n_i \in N_S(u)} f(n_i) \cdot f(u) \right]$$

where $N_S(u)$ is the network neighborhood of $u$ generated by the biased random walk strategy $S$.

### Comparison: DeepWalk vs Node2Vec

| Feature | DeepWalk | Node2Vec |
|---------|----------|----------|
| Walk type | Uniform random | Biased (p, q parameters) |
| Exploration strategy | Fixed (unbiased) | Flexible BFS/DFS interpolation |
| Structural equivalence | Limited | Yes (high q) |
| Homophily | Yes (default behavior) | Yes (low q) |
| Negative sampling | Hierarchical softmax | Negative sampling (more efficient) |
| Parameters | γ, t, w, d | γ, t, w, d, **p, q** |
| Expressiveness | Good | Better (more flexible) |

In [0]:
# ============================================================
# Node2Vec Implementation from Scratch
# ============================================================

class Node2Vec:
    """
    Node2Vec: Scalable Feature Learning for Networks
    
    Extends DeepWalk with biased random walks controlled by:
    - p (return parameter): controls backtracking
    - q (in-out parameter): controls BFS vs DFS behavior
    """
    
    def __init__(self, graph, walk_length=40, num_walks=10,
                 embedding_dim=64, window_size=5, p=1.0, q=1.0,
                 workers=4, seed=42):
        """
        Parameters:
        -----------
        graph : networkx.Graph
            Input graph
        p : float
            Return parameter. Low p → more backtracking (local)
        q : float  
            In-out parameter. Low q → DFS-like (homophily),
                              High q → BFS-like (structural)
        """
        self.graph = graph
        self.walk_length = walk_length
        self.num_walks = num_walks
        self.embedding_dim = embedding_dim
        self.window_size = window_size
        self.p = p
        self.q = q
        self.workers = workers
        self.seed = seed
        self.model = None
        
        # Precompute transition probabilities for efficiency
        self._precompute_transition_probs()
        
    def _precompute_transition_probs(self):
        """
        Precompute the biased transition probabilities for all edges.
        
        For edge (t, v), compute α_pq(t, x) for all neighbors x of v:
          - α = 1/p if x == t  (return to previous)
          - α = 1   if x is neighbor of t (stay close)  
          - α = 1/q if x is NOT neighbor of t (move away)
        """
        self.alias_nodes = {}
        self.alias_edges = {}
        
        # For the first step (no previous node), use uniform distribution
        for node in self.graph.nodes():
            neighbors = list(self.graph.neighbors(node))
            if len(neighbors) > 0:
                # Uniform probabilities for first step
                probs = [1.0 / len(neighbors)] * len(neighbors)
                self.alias_nodes[node] = self._create_alias_table(probs)
            else:
                self.alias_nodes[node] = ([], [])
        
        # For subsequent steps, compute biased probabilities
        for edge in self.graph.edges():
            self.alias_edges[edge] = self._compute_edge_probs(edge[0], edge[1])
            if not self.graph.is_directed():
                self.alias_edges[(edge[1], edge[0])] = self._compute_edge_probs(edge[1], edge[0])
                
    def _compute_edge_probs(self, t, v):
        """
        Compute transition probabilities from v, having come from t.
        
        α_pq(t, x) = 1/p if d_tx=0, 1 if d_tx=1, 1/q if d_tx=2
        """
        neighbors_v = list(self.graph.neighbors(v))
        neighbors_t = set(self.graph.neighbors(t))
        
        unnormalized_probs = []
        for x in neighbors_v:
            weight = self.graph[v][x].get('weight', 1.0)
            
            if x == t:
                # d_tx = 0: returning to previous node
                unnormalized_probs.append(weight / self.p)
            elif x in neighbors_t:
                # d_tx = 1: x is also a neighbor of t
                unnormalized_probs.append(weight)
            else:
                # d_tx = 2: x is NOT a neighbor of t (moving away)
                unnormalized_probs.append(weight / self.q)
        
        # Normalize
        prob_sum = sum(unnormalized_probs)
        normalized_probs = [p / prob_sum for p in unnormalized_probs]
        
        return self._create_alias_table(normalized_probs)
    
    def _create_alias_table(self, probs):
        """
        Create alias table for O(1) sampling from discrete distribution.
        (Vose's Alias Method)
        """
        K = len(probs)
        q_table = np.zeros(K)
        J = np.zeros(K, dtype=int)
        
        smaller = []
        larger = []
        
        for kk, prob in enumerate(probs):
            q_table[kk] = K * prob
            if q_table[kk] < 1.0:
                smaller.append(kk)
            else:
                larger.append(kk)
        
        while len(smaller) > 0 and len(larger) > 0:
            small = smaller.pop()
            large = larger.pop()
            
            J[small] = large
            q_table[large] = q_table[large] + q_table[small] - 1.0
            
            if q_table[large] < 1.0:
                smaller.append(large)
            else:
                larger.append(large)
        
        return (J, q_table)
    
    def _alias_sample(self, J, q):
        """Sample from alias table in O(1) time."""
        K = len(J)
        kk = int(np.floor(np.random.rand() * K))
        if np.random.rand() < q[kk]:
            return kk
        else:
            return J[kk]
    
    def _biased_walk(self, start_node):
        """
        Generate a single biased random walk.
        First step is uniform; subsequent steps use biased probabilities.
        """
        walk = [start_node]
        neighbors = list(self.graph.neighbors(start_node))
        
        if len(neighbors) == 0:
            return walk
        
        # First step: uniform random
        J, q = self.alias_nodes[start_node]
        if len(J) > 0:
            first_step = neighbors[self._alias_sample(J, q)]
            walk.append(first_step)
        else:
            return walk
        
        # Subsequent steps: biased by p, q
        for _ in range(self.walk_length - 2):
            cur = walk[-1]
            prev = walk[-2]
            cur_neighbors = list(self.graph.neighbors(cur))
            
            if len(cur_neighbors) == 0:
                break
            
            edge = (prev, cur)
            if edge in self.alias_edges:
                J, q = self.alias_edges[edge]
                next_node = cur_neighbors[self._alias_sample(J, q)]
                walk.append(next_node)
            else:
                # Fallback to uniform if edge not found
                walk.append(random.choice(cur_neighbors))
        
        return walk
    
    def _generate_walks(self):
        """Generate biased random walks for all nodes."""
        walks = []
        nodes = list(self.graph.nodes())
        
        for _ in range(self.num_walks):
            random.shuffle(nodes)
            for node in nodes:
                walk = self._biased_walk(node)
                walks.append([str(n) for n in walk])
                
        return walks
    
    def fit(self):
        """Train Node2Vec model."""
        np.random.seed(self.seed)
        random.seed(self.seed)
        
        print(f"Node2Vec parameters: p={self.p}, q={self.q}")
        print(f"Generating {self.num_walks} biased walks per node...")
        walks = self._generate_walks()
        
        print(f"Training Word2Vec Skip-gram (dim={self.embedding_dim}, "
              f"window={self.window_size})...")
        self.model = Word2Vec(
            sentences=walks,
            vector_size=self.embedding_dim,
            window=self.window_size,
            min_count=0,
            sg=1,         # Skip-gram
            hs=0,         # Use negative sampling (Node2Vec paper)
            negative=5,   # Number of negative samples
            workers=self.workers,
            seed=self.seed,
            epochs=5
        )
        
        print(f"Node2Vec training complete! "
              f"Learned embeddings for {len(self.model.wv)} nodes.")
        return self
    
    def get_embedding(self, node):
        """Get embedding vector for a specific node."""
        return self.model.wv[str(node)]
    
    def get_all_embeddings(self):
        """Get embedding matrix for all nodes."""
        nodes = sorted(self.graph.nodes())
        embeddings = np.array([self.model.wv[str(n)] for n in nodes])
        return nodes, embeddings
    
    def most_similar(self, node, topn=10):
        """Find most similar nodes in embedding space."""
        return self.model.wv.most_similar(str(node), topn=topn)


print("✅ Node2Vec class defined successfully")

In [0]:
# ============================================================
# Create a realistic sample graph for demonstrations
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns

# Load Zachary's Karate Club (34 nodes, 78 edges, 2 communities)
G = nx.karate_club_graph()

# Add community labels as node attributes
community_map = {}
for node in G.nodes():
    club = G.nodes[node]['club']
    community_map[node] = 0 if club == 'Mr. Hi' else 1

nx.set_node_attributes(G, community_map, 'community')

print(f"Graph Statistics:")
print(f"  Nodes: {G.number_of_nodes()}")
print(f"  Edges: {G.number_of_edges()}")
print(f"  Average degree: {2 * G.number_of_edges() / G.number_of_nodes():.2f}")
print(f"  Communities: {len(set(community_map.values()))}")
print(f"  Community 0 (Mr. Hi): {sum(1 for v in community_map.values() if v == 0)} nodes")
print(f"  Community 1 (Officer): {sum(1 for v in community_map.values() if v == 1)} nodes")

# Visualize the graph
fig, ax = plt.subplots(1, 1, figsize=(10, 8))
pos = nx.spring_layout(G, seed=42)
colors = [community_map[n] for n in G.nodes()]
nx.draw(G, pos, node_color=colors, cmap=plt.cm.Set1, 
        with_labels=True, node_size=500, font_size=10, 
        font_weight='bold', edge_color='gray', alpha=0.9, ax=ax)
ax.set_title("Zachary's Karate Club \u2014 Ground Truth Communities", fontsize=14)
plt.tight_layout()
plt.show()

In [0]:
# ============================================================
# Train DeepWalk
# ============================================================
print("=" * 60)
print("TRAINING DEEPWALK")
print("=" * 60)

dw = DeepWalk(
    graph=G,
    walk_length=40,
    num_walks=80,
    embedding_dim=64,
    window_size=5,
    seed=42
)
dw.fit()

print("\n" + "=" * 60)
print("TRAINING NODE2VEC (Homophily-focused: low q)")
print("=" * 60)

# p=1, q=0.5: DFS-like, captures community/homophily
n2v_homophily = Node2Vec(
    graph=G,
    walk_length=40,
    num_walks=80,
    embedding_dim=64,
    window_size=5,
    p=1.0,
    q=0.5,
    seed=42
)
n2v_homophily.fit()

print("\n" + "=" * 60)
print("TRAINING NODE2VEC (Structure-focused: high q)")
print("=" * 60)

# p=1, q=2: BFS-like, captures structural equivalence
n2v_structural = Node2Vec(
    graph=G,
    walk_length=40,
    num_walks=80,
    embedding_dim=64,
    window_size=5,
    p=1.0,
    q=2.0,
    seed=42
)
n2v_structural.fit()

print("\n\u2705 All models trained successfully!")

---
# Downstream Tasks Using Node Embeddings

Once we have learned dense vector representations of nodes via DeepWalk or Node2Vec, these embeddings become **universal features** that can be fed into virtually any machine learning pipeline.

$$\text{Graph} \xrightarrow{\text{DeepWalk/Node2Vec}} \text{Embeddings} \in \mathbb{R}^{|V| \times d} \xrightarrow{\text{ML Algorithm}} \text{Predictions}$$

Below we demonstrate **15 downstream applications** using the embeddings we just trained.

## Task 1: Node Classification

### Description
Node classification assigns categorical labels to nodes in a graph based on their learned embedding representations. The core idea is that nodes with similar structural roles or neighborhood patterns will have similar embeddings, making them amenable to standard supervised classifiers.

**Real-world applications:** Fraud vs. non-fraud customers, spam vs. genuine accounts, protein function prediction, paper topic classification.

### Input Data
- **Features:** Node embeddings $\Phi(v) \in \mathbb{R}^d$ learned via DeepWalk/Node2Vec
- **Labels:** Ground-truth categories $y(v) \in \{0, 1, \ldots, C-1\}$ for a subset of nodes
- **Split:** Stratified train/test partition preserving class balance

### Mathematical Formulation

**Objective:** Learn a mapping $f: \mathbb{R}^d \rightarrow \{0, \ldots, C-1\}$ that minimizes classification error.

**Cross-entropy loss** (for multi-class):
$$\mathcal{L} = -\sum_{v \in V_{\text{train}}} \sum_{c=0}^{C-1} y_c(v) \cdot \log\left(\hat{y}_c(v)\right)$$

where $\hat{y}_c(v) = \text{softmax}\left(W \cdot \Phi(v) + b\right)_c$ for Logistic Regression, or a non-linear function for tree-based models.

**For Logistic Regression:**
$$P(y=c \mid v) = \frac{\exp(w_c^\top \Phi(v) + b_c)}{\sum_{k=0}^{C-1} \exp(w_k^\top \Phi(v) + b_k)}$$

### Expected Outcome
- Accuracy and F1-score per classifier (Logistic Regression, Random Forest, Gradient Boosting)
- Comparison of DeepWalk vs. Node2Vec embedding quality for classification
- Demonstration that structural proximity in the graph translates to label agreement

## Task 2: Link Prediction

### Description
Link prediction determines whether an edge exists (or will form) between two nodes. Given learned node embeddings, we construct **edge-level features** from pairs of node vectors and train a binary classifier.

**Real-world applications:** Friend recommendations (social networks), protein-protein interaction prediction, co-purchase prediction, knowledge graph completion.

### Input Data
- **Positive samples:** Existing edges $(u, v) \in E$ — labeled 1
- **Negative samples:** Non-edges $(u, v) \notin E$ — labeled 0
- **Edge features:** Computed from node embedding pairs using various operators

### Mathematical Formulation

**Edge feature operators** for a pair $(u, v)$:

| Operator | Formula | Intuition |
|----------|---------|----------|
| Hadamard | $\Phi(u) \odot \Phi(v)$ | Element-wise interaction |
| Concatenation | $[\Phi(u) \| \Phi(v)]$ | Full information preservation |
| L1 Distance | $|\Phi(u) - \Phi(v)|$ | Absolute difference |
| L2 Distance | $(\Phi(u) - \Phi(v))^2$ | Squared difference |
| Cosine Similarity | $\frac{\Phi(u) \cdot \Phi(v)}{\|\Phi(u)\| \cdot \|\Phi(v)\|}$ | Angular proximity |

**Binary classification objective:**
$$\mathcal{L} = -\sum_{(u,v)} \left[ y_{uv} \log \sigma(w^\top \text{op}(u,v)) + (1-y_{uv}) \log(1 - \sigma(w^\top \text{op}(u,v))) \right]$$

where $\sigma$ is the sigmoid function and $\text{op}(u,v)$ is the chosen edge operator.

### Expected Outcome
- ROC-AUC and Average Precision scores
- Comparison of edge feature operators (Hadamard typically wins)
- Demonstration that embeddings capture latent edge likelihood

## Task 3: Community Detection / Clustering

### Description
Community detection groups nodes into clusters where intra-cluster connections are dense and inter-cluster connections are sparse. By operating on node embeddings rather than the adjacency matrix directly, we can leverage standard clustering algorithms.

**Real-world applications:** Social circle discovery, market segmentation, biological module detection, network partitioning.

### Input Data
- **Features:** Node embedding matrix $X = [\Phi(v_1), \ldots, \Phi(v_n)]^\top \in \mathbb{R}^{n \times d}$
- **Ground truth:** Known community labels (for evaluation)
- **No labels required** for the clustering itself (unsupervised)

### Mathematical Formulation

**K-Means objective:**
$$\min_{C_1, \ldots, C_k} \sum_{i=1}^{k} \sum_{v \in C_i} \|\Phi(v) - \mu_i\|^2, \quad \mu_i = \frac{1}{|C_i|} \sum_{v \in C_i} \Phi(v)$$

**HDBSCAN** uses mutual reachability distance:
$$d_{\text{mreach}}(a, b) = \max\left(\text{core}_k(a),\ \text{core}_k(b),\ d(a, b)\right)$$

where $\text{core}_k(a)$ is the distance to the $k$-th nearest neighbor of $a$.

**Evaluation metrics:**
- NMI (Normalized Mutual Information): $\text{NMI}(Y, \hat{Y}) = \frac{2 \cdot I(Y; \hat{Y})}{H(Y) + H(\hat{Y})}$
- ARI (Adjusted Rand Index): corrects for random chance agreement
- Silhouette Score: $s(v) = \frac{b(v) - a(v)}{\max(a(v), b(v))}$ where $a$ = intra-cluster, $b$ = nearest-cluster distance

### Expected Outcome
- Clustering quality metrics (NMI, ARI, Silhouette) for K-Means, DBSCAN, HDBSCAN, Spectral
- Visual comparison of discovered clusters vs. ground truth communities
- Demonstration that embedding space naturally separates communities

## Task 4: Recommendation Systems

### Description
Recommendation systems suggest relevant items (users, products, content) to a query entity. Using graph embeddings, we recommend nodes that are **closest in embedding space** — capturing both direct connections and higher-order structural similarity.

**Real-world applications:** Friend suggestions, product recommendations, movie/song recommendations, content personalization.

### Input Data
- **Query:** A node $u$ for which we want recommendations
- **Candidate pool:** All other nodes $V \setminus \{u\}$
- **Embeddings:** Pre-trained vectors $\Phi(v) \in \mathbb{R}^d$ for all $v \in V$

### Mathematical Formulation

**Top-k recommendation** for query node $u$:
$$R_k(u) = \underset{v \in V \setminus \{u\}}{\text{arg top-}k}\ \text{sim}(\Phi(u), \Phi(v))$$

**Similarity measures:**
$$\text{Cosine:} \quad \text{sim}(u, v) = \frac{\Phi(u) \cdot \Phi(v)}{\|\Phi(u)\| \cdot \|\Phi(v)\|}$$
$$\text{Dot product:} \quad \text{sim}(u, v) = \Phi(u)^\top \Phi(v)$$
$$\text{Euclidean (inverse):} \quad \text{sim}(u, v) = \frac{1}{1 + \|\Phi(u) - \Phi(v)\|_2}$$

**Why this works:** The Skip-gram objective ensures nodes appearing in similar random walk contexts have similar embeddings. Thus, structurally proximate nodes (same community, shared neighbors) score highly.

### Expected Outcome
- Top-k recommendations for key nodes with similarity scores
- Evaluation: what fraction of recommendations share the same community
- Comparison of DeepWalk vs. Node2Vec recommendation quality

## Task 5: Similarity Search / Nearest Neighbor Search

### Description
Similarity search retrieves the most similar nodes to a query by computing distances in the embedding space. For large graphs, **approximate nearest neighbor (ANN)** methods like FAISS provide sub-linear query time.

**Real-world applications:** Find-similar-users, duplicate detection, content retrieval, real-time recommendations at scale.

### Input Data
- **Index:** All node embeddings $\Phi(v) \in \mathbb{R}^d$, normalized for cosine similarity
- **Query:** Embedding vector of the target node
- **k:** Number of nearest neighbors to retrieve

### Mathematical Formulation

**Exact k-NN problem:**
$$\text{NN}_k(q) = \underset{S \subseteq V,\ |S|=k}{\arg\min} \sum_{v \in S} \|\Phi(q) - \Phi(v)\|_2$$

**FAISS IVF (Inverted File Index):**
1. Partition embedding space into $n_{\text{list}}$ Voronoi cells via K-Means
2. At query time, probe only $n_{\text{probe}}$ nearest cells
3. Complexity: $O\left(\frac{n}{n_{\text{list}}} \cdot n_{\text{probe}} \cdot d\right)$ vs. brute-force $O(n \cdot d)$

**Cosine similarity via inner product** (on L2-normalized vectors):
$$\text{cosine}(u, v) = \Phi(u)^\top \Phi(v) \quad \text{when } \|\Phi(u)\| = \|\Phi(v)\| = 1$$

### Expected Outcome
- Exact vs. approximate nearest neighbor results (quality comparison)
- Latency benchmark: Flat index vs. IVF index
- Demonstration of FAISS for scalable graph node retrieval

## Task 6: Graph Visualization

### Description
Graph visualization projects high-dimensional node embeddings into 2D or 3D for human inspection. Different dimensionality reduction methods reveal different aspects of graph structure.

**Real-world applications:** Network exploration, community structure visualization, outlier identification, presentation of analysis results.

### Input Data
- **Embeddings:** $X \in \mathbb{R}^{n \times d}$ (e.g., 34 nodes × 64 dimensions)
- **Labels:** Community assignments for color-coding
- **Multiple models:** DeepWalk, Node2Vec (homophily), Node2Vec (structural)

### Mathematical Formulation

**PCA** (linear, global structure):
$$X_{2D} = X \cdot W_2, \quad W_2 = [w_1, w_2] \text{ (top-2 eigenvectors of } X^\top X\text{)}$$
$$\text{Maximize: } \text{Var}(Xw) = w^\top X^\top X w \text{ subject to } \|w\| = 1$$

**t-SNE** (non-linear, local structure):
$$\text{Minimize: } KL(P \| Q) = \sum_{i \neq j} p_{ij} \log \frac{p_{ij}}{q_{ij}}$$
where $p_{ij} = \frac{\exp(-\|x_i - x_j\|^2 / 2\sigma_i^2)}{\sum_{k \neq l} \exp(-\|x_k - x_l\|^2 / 2\sigma_k^2)}$ (high-D affinities)
and $q_{ij} = \frac{(1 + \|y_i - y_j\|^2)^{-1}}{\sum_{k \neq l}(1 + \|y_k - y_l\|^2)^{-1}}$ (low-D Student-t kernel)

**UMAP** (non-linear, preserves both local and global):
- Constructs a fuzzy simplicial set from high-D data
- Optimizes a low-D representation minimizing cross-entropy between fuzzy sets

### Expected Outcome
- Side-by-side comparison of PCA, t-SNE, UMAP projections
- Visual evidence that Node2Vec (homophily) separates communities better
- Comparison of all three trained models' embedding quality

## Task 7: Feature Engineering for Machine Learning Models

### Description
Node embeddings can be combined with traditional hand-crafted graph features (degree centrality, betweenness, PageRank) to create richer feature representations. This often outperforms either feature set alone.

**Real-world applications:** Enriching fraud detection models, improving churn prediction, boosting any graph-aware ML pipeline.

### Input Data
- **Structural features** (hand-crafted): degree, betweenness centrality, closeness centrality, clustering coefficient, PageRank
- **Embedding features:** $\Phi(v) \in \mathbb{R}^d$ from Node2Vec
- **Combined:** $x(v) = [f_{\text{structural}}(v) \| \Phi(v)] \in \mathbb{R}^{5+d}$

### Mathematical Formulation

**Traditional graph features:**
$$\text{Degree}(v) = |\mathcal{N}(v)|$$
$$\text{Betweenness}(v) = \sum_{s \neq v \neq t} \frac{\sigma_{st}(v)}{\sigma_{st}}$$
$$\text{Closeness}(v) = \frac{n-1}{\sum_{u \neq v} d(v, u)}$$
$$\text{Clustering}(v) = \frac{2 |\{(i,j) : i,j \in \mathcal{N}(v), (i,j) \in E\}|}{|\mathcal{N}(v)| \cdot (|\mathcal{N}(v)| - 1)}$$
$$\text{PageRank}(v) = \frac{1-d}{n} + d \sum_{u \in \text{in}(v)} \frac{\text{PR}(u)}{|\text{out}(u)|}$$

**Hypothesis:** Combined features capture complementary information:
- Structural features: explicit local topology
- Embeddings: implicit higher-order relationships

### Expected Outcome
- Cross-validated accuracy for: structural-only, embedding-only, combined
- Feature importance ranking showing which features matter most
- Evidence that combined features outperform either alone

## Task 8: Anomaly Detection

### Description
Anomaly detection identifies nodes whose embeddings deviate significantly from their neighbors or from expected patterns. Anomalous nodes may represent fraudulent entities, compromised accounts, or unusual structural positions.

**Real-world applications:** Fraud detection, network intrusion detection, bot identification, detecting compromised nodes in IoT networks.

### Input Data
- **Embeddings:** Node vectors $\Phi(v) \in \mathbb{R}^d$
- **Graph structure:** Adjacency information for neighborhood comparison
- **No labels required** (unsupervised anomaly detection)

### Mathematical Formulation

**Method 1 — Isolation Forest:**
$$\text{Score}(v) = 2^{-E[h(v)] / c(n)}$$
where $h(v)$ = average path length to isolate $v$ in random trees, $c(n)$ = average path length in a BST with $n$ samples. Anomalies have shorter paths (easier to isolate).

**Method 2 — Local Outlier Factor (LOF):**
$$\text{LOF}_k(v) = \frac{1}{|N_k(v)|} \sum_{u \in N_k(v)} \frac{\text{lrd}_k(u)}{\text{lrd}_k(v)}$$
where $\text{lrd}_k(v) = 1 / \left(\frac{1}{|N_k(v)|} \sum_{u \in N_k(v)} \text{reach-dist}_k(v, u)\right)$

LOF > 1 indicates lower density than neighbors (potential anomaly).

**Method 3 — Neighborhood Deviation:**
$$\text{deviation}(v) = \left\|\Phi(v) - \frac{1}{|\mathcal{N}(v)|} \sum_{u \in \mathcal{N}(v)} \Phi(u)\right\|_2$$

Nodes at community boundaries will have high deviation since their neighbors span different regions in embedding space.

### Expected Outcome
- Anomaly scores from three methods (Isolation Forest, LOF, Deviation)
- Identification of boundary/bridge nodes as natural anomalies
- Visualization of anomaly scores in 2D embedding space

In [0]:
# ============================================================
# TASK 1: NODE CLASSIFICATION
# ============================================================
# Predict labels for nodes using embeddings as features.
# Use case: fraud detection, spam classification, user type prediction
# 
# Mathematical formulation:
#   Given embeddings Φ(v) ∈ ℝ^d and labels y(v) ∈ {0, 1, ..., C-1}
#   Train classifier: f(Φ(v)) → ŷ(v)
#   Minimize: L = -Σ y_c · log(f_c(Φ(v))) (cross-entropy)
# ============================================================

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Get embeddings and labels
nodes, embeddings_dw = dw.get_all_embeddings()
nodes, embeddings_n2v = n2v_homophily.get_all_embeddings()
labels = np.array([community_map[n] for n in nodes])

# Train/test split (stratified to maintain class balance)
X_train, X_test, y_train, y_test = train_test_split(
    embeddings_n2v, labels, test_size=0.3, random_state=42, stratify=labels
)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("=" * 60)
print("TASK 1: NODE CLASSIFICATION")
print("=" * 60)
print(f"\nTraining samples: {len(X_train)}, Test samples: {len(X_test)}")
print(f"Class distribution (train): {dict(zip(*np.unique(y_train, return_counts=True)))}")

# Compare multiple classifiers
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results = []
for name, clf in classifiers.items():
    clf.fit(X_train_scaled, y_train)
    y_pred = clf.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    results.append({'Classifier': name, 'Accuracy': acc, 'F1-Score': f1})
    print(f"\n{name}:")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  F1-Score: {f1:.4f}")

# Compare DeepWalk vs Node2Vec for classification
print("\n" + "-" * 60)
print("DeepWalk vs Node2Vec Comparison:")
for emb_name, emb in [('DeepWalk', embeddings_dw), ('Node2Vec (homophily)', embeddings_n2v)]:
    X_tr, X_te, y_tr, y_te = train_test_split(
        emb, labels, test_size=0.3, random_state=42, stratify=labels
    )
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, clf.predict(X_te))
    print(f"  {emb_name}: Accuracy = {acc:.4f}")

In [0]:
# ============================================================
# TASK 2: LINK PREDICTION
# ============================================================
# Predict missing or future edges between nodes.
# Use case: friend recommendations, co-purchase prediction
#
# Mathematical formulation:
#   For a pair (u, v), define edge features:
#   - Hadamard product: Φ(u) ⊙ Φ(v)
#   - Concatenation: [Φ(u) || Φ(v)]
#   - L1 distance: |Φ(u) - Φ(v)|
#   - L2 distance: ||Φ(u) - Φ(v)||₂
#   - Cosine similarity: Φ(u)·Φ(v) / (||Φ(u)|| · ||Φ(v)||)
#
#   Train binary classifier: f(edge_features) → {0, 1}
# ============================================================

from sklearn.metrics import roc_auc_score, average_precision_score
from itertools import combinations

print("=" * 60)
print("TASK 2: LINK PREDICTION")
print("=" * 60)

# Create positive samples (existing edges)
edges = list(G.edges())
positive_edges = random.sample(edges, min(len(edges), 50))

# Create negative samples (non-existing edges)
non_edges = []
nodes_list = list(G.nodes())
while len(non_edges) < len(positive_edges):
    u, v = random.choice(nodes_list), random.choice(nodes_list)
    if u != v and not G.has_edge(u, v) and (u, v) not in non_edges:
        non_edges.append((u, v))

# Define edge feature operators
def hadamard(u, v, model):
    """Element-wise product: Φ(u) ⊙ Φ(v)"""
    return model.get_embedding(u) * model.get_embedding(v)

def l1_distance(u, v, model):
    """|Φ(u) - Φ(v)|"""
    return np.abs(model.get_embedding(u) - model.get_embedding(v))

def l2_distance(u, v, model):
    """||Φ(u) - Φ(v)||₂ (element-wise squared diff)"""
    return (model.get_embedding(u) - model.get_embedding(v)) ** 2

def cosine_sim(u, v, model):
    """Cosine similarity as a scalar feature"""
    e_u = model.get_embedding(u)
    e_v = model.get_embedding(v)
    return np.array([np.dot(e_u, e_v) / (np.linalg.norm(e_u) * np.linalg.norm(e_v) + 1e-8)])

# Build dataset using Hadamard product (most common in literature)
X_pos = np.array([hadamard(u, v, n2v_homophily) for u, v in positive_edges])
X_neg = np.array([hadamard(u, v, n2v_homophily) for u, v in non_edges])

X_link = np.vstack([X_pos, X_neg])
y_link = np.array([1] * len(X_pos) + [0] * len(X_neg))

# Train link prediction model
X_tr, X_te, y_tr, y_te = train_test_split(
    X_link, y_link, test_size=0.3, random_state=42, stratify=y_link
)

lr_link = LogisticRegression(max_iter=1000, random_state=42)
lr_link.fit(X_tr, y_tr)

y_prob = lr_link.predict_proba(X_te)[:, 1]
auc = roc_auc_score(y_te, y_prob)
ap = average_precision_score(y_te, y_prob)

print(f"\nLink Prediction Results (Hadamard + Logistic Regression):")
print(f"  ROC-AUC: {auc:.4f}")
print(f"  Average Precision: {ap:.4f}")

# Compare operators
print("\nComparing Edge Feature Operators:")
for op_name, op_func in [('Hadamard', hadamard), ('L1 Distance', l1_distance), ('L2 Distance', l2_distance)]:
    X_p = np.array([op_func(u, v, n2v_homophily) for u, v in positive_edges])
    X_n = np.array([op_func(u, v, n2v_homophily) for u, v in non_edges])
    X = np.vstack([X_p, X_n])
    y = np.array([1]*len(X_p) + [0]*len(X_n))
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    clf = LogisticRegression(max_iter=1000, random_state=42).fit(Xtr, ytr)
    score = roc_auc_score(yte, clf.predict_proba(Xte)[:, 1])
    print(f"  {op_name}: AUC = {score:.4f}")

In [0]:
# ============================================================
# TASK 3: COMMUNITY DETECTION / CLUSTERING
# ============================================================
# Group structurally similar nodes into communities.
# Use case: social circles, protein functional groups, market segments
#
# Mathematical formulation:
#   Given embeddings X = {Φ(v₁), ..., Φ(vₙ)} ∈ ℝ^(n×d)
#   Find clusters C₁, ..., C_k that minimize intra-cluster distance:
#   
#   K-Means: min_C Σᵢ Σ_{v∈Cᵢ} ||Φ(v) - μᵢ||²
#   where μᵢ = (1/|Cᵢ|) Σ_{v∈Cᵢ} Φ(v) is the centroid
#
#   HDBSCAN: density-based, finds clusters of varying density
#   Uses mutual reachability distance:
#   d_mreach(a, b) = max(core_k(a), core_k(b), d(a,b))
# ============================================================

from sklearn.cluster import KMeans, DBSCAN, SpectralClustering
from sklearn.metrics import (
    normalized_mutual_info_score, adjusted_rand_score, 
    silhouette_score, homogeneity_score
)
import hdbscan

print("=" * 60)
print("TASK 3: COMMUNITY DETECTION / CLUSTERING")
print("=" * 60)

nodes, embeddings = n2v_homophily.get_all_embeddings()
true_labels = np.array([community_map[n] for n in nodes])

# Standardize embeddings for clustering
scaler_clust = StandardScaler()
X_scaled = scaler_clust.fit_transform(embeddings)

# K-Means Clustering
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)

# DBSCAN
dbscan = DBSCAN(eps=1.5, min_samples=3)
dbscan_labels = dbscan.fit_predict(X_scaled)

# HDBSCAN (density-based hierarchical)
hdb = hdbscan.HDBSCAN(min_cluster_size=5, min_samples=3)
hdb_labels = hdb.fit_predict(X_scaled)

# Spectral Clustering
spectral = SpectralClustering(n_clusters=2, random_state=42, affinity='nearest_neighbors')
spectral_labels = spectral.fit_predict(X_scaled)

# Evaluation metrics
print("\nClustering Results vs Ground Truth:")
print(f"{'Method':<20} {'NMI':>8} {'ARI':>8} {'Homogeneity':>12} {'Silhouette':>11}")
print("-" * 65)

for name, pred_labels in [('K-Means', kmeans_labels), 
                           ('DBSCAN', dbscan_labels),
                           ('HDBSCAN', hdb_labels),
                           ('Spectral', spectral_labels)]:
    # Filter out noise labels (-1) for NMI/ARI
    mask = pred_labels >= 0
    if mask.sum() > 0:
        nmi = normalized_mutual_info_score(true_labels[mask], pred_labels[mask])
        ari = adjusted_rand_score(true_labels[mask], pred_labels[mask])
        homo = homogeneity_score(true_labels[mask], pred_labels[mask])
        if len(set(pred_labels[mask])) > 1:
            sil = silhouette_score(X_scaled[mask], pred_labels[mask])
        else:
            sil = -1.0
        print(f"{name:<20} {nmi:>8.4f} {ari:>8.4f} {homo:>12.4f} {sil:>11.4f}")
    else:
        print(f"{name:<20} {'N/A':>8} {'N/A':>8} {'N/A':>12} {'N/A':>11}")

# Visualize clusters using PCA for 2D projection
from sklearn.decomposition import PCA
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Ground truth
axes[0].scatter(X_2d[:, 0], X_2d[:, 1], c=true_labels, cmap='Set1', s=100, edgecolors='black', linewidths=0.5)
axes[0].set_title('Ground Truth Communities', fontsize=12)

# K-Means
axes[1].scatter(X_2d[:, 0], X_2d[:, 1], c=kmeans_labels, cmap='Set1', s=100, edgecolors='black', linewidths=0.5)
axes[1].set_title('K-Means on Node2Vec Embeddings', fontsize=12)

# HDBSCAN
axes[2].scatter(X_2d[:, 0], X_2d[:, 1], c=hdb_labels, cmap='Set1', s=100, edgecolors='black', linewidths=0.5)
axes[2].set_title('HDBSCAN on Node2Vec Embeddings', fontsize=12)

for ax in axes:
    ax.set_xlabel('PCA Component 1')
    ax.set_ylabel('PCA Component 2')
    
plt.tight_layout()
plt.show()

In [0]:
# ============================================================
# TASK 4: RECOMMENDATION SYSTEMS
# ============================================================
# Recommend similar nodes (users, products, items) based on
# proximity in embedding space.
# Use case: friend suggestions, product recommendations
#
# Mathematical formulation:
#   For a query node u, recommend top-k nodes:
#   R(u) = argmax_{v ∈ V\{u}} sim(Φ(u), Φ(v))
#
#   Common similarity measures:
#   - Cosine: sim(u,v) = Φ(u)·Φ(v) / (||Φ(u)|| · ||Φ(v)||)
#   - Dot product: sim(u,v) = Φ(u)·Φ(v)
#   - Euclidean (inverse): sim(u,v) = 1 / (1 + ||Φ(u) - Φ(v)||)
# ============================================================

from scipy.spatial.distance import cosine, euclidean

print("=" * 60)
print("TASK 4: RECOMMENDATION SYSTEMS")
print("=" * 60)

def recommend_nodes(model, query_node, top_k=5, method='cosine'):
    """
    Recommend top-k most similar nodes to query_node.
    
    Parameters:
    -----------
    model : DeepWalk or Node2Vec trained model
    query_node : int, the node to get recommendations for
    top_k : int, number of recommendations
    method : str, similarity measure ('cosine', 'dot', 'euclidean')
    """
    query_emb = model.get_embedding(query_node)
    nodes_sorted = sorted(model.graph.nodes())
    
    similarities = []
    for node in nodes_sorted:
        if node == query_node:
            continue
        node_emb = model.get_embedding(node)
        
        if method == 'cosine':
            sim = 1 - cosine(query_emb, node_emb)  # cosine returns distance
        elif method == 'dot':
            sim = np.dot(query_emb, node_emb)
        elif method == 'euclidean':
            sim = 1.0 / (1.0 + euclidean(query_emb, node_emb))
        
        similarities.append((node, sim))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_k]

# Demo: Recommend nodes for key nodes in the Karate Club
print("\nRecommendations based on Node2Vec embeddings:")
print("(Using cosine similarity in embedding space)\n")

for query in [0, 33, 2, 13]:  # Hub nodes from each community
    recs = recommend_nodes(n2v_homophily, query, top_k=5, method='cosine')
    actual_neighbors = set(G.neighbors(query))
    community = community_map[query]
    
    print(f"Node {query} (Community: {community}, Degree: {G.degree(query)}):")
    print(f"  Actual neighbors: {sorted(actual_neighbors)}")
    print(f"  Top-5 recommendations:")
    for node, sim in recs:
        is_neighbor = "✓ neighbor" if node in actual_neighbors else "✗ new"
        same_comm = "same community" if community_map[node] == community else "diff community"
        print(f"    Node {node:2d} (sim={sim:.4f}) [{is_neighbor}, {same_comm}]")
    print()

# Evaluate: How often do recommendations match actual community?
print("\nRecommendation Quality (Community Hit Rate):")
for model_name, model in [('DeepWalk', dw), ('Node2Vec', n2v_homophily)]:
    hits = 0
    total = 0
    for node in G.nodes():
        recs = recommend_nodes(model, node, top_k=5, method='cosine')
        node_comm = community_map[node]
        for rec_node, _ in recs:
            total += 1
            if community_map[rec_node] == node_comm:
                hits += 1
    print(f"  {model_name}: {hits}/{total} = {hits/total:.2%} same-community recommendations")

In [0]:
# ============================================================
# TASK 5: SIMILARITY SEARCH / NEAREST NEIGHBOR SEARCH
# ============================================================
# Retrieve most similar nodes using efficient vector search.
# Use case: find-similar-users, duplicate detection, retrieval
#
# Mathematical formulation:
#   Exact k-NN: For query q, find k nearest neighbors:
#   NN_k(q) = argmin_{v ∈ V, |S|=k} ||Φ(q) - Φ(v)||₂
#
#   FAISS uses IVF (Inverted File Index) + PQ (Product Quantization)
#   for approximate nearest neighbors (ANN) in sublinear time.
#
#   Complexity:
#   - Brute force: O(n·d) per query
#   - FAISS IVF: O(n/nprobe · d) per query
#   - FAISS HNSW: O(log n · d) per query
# ============================================================

import faiss

print("=" * 60)
print("TASK 5: SIMILARITY SEARCH (FAISS)")
print("=" * 60)

# Get all embeddings
nodes, embeddings = n2v_homophily.get_all_embeddings()
embeddings_float32 = embeddings.astype(np.float32)

# Normalize for cosine similarity (FAISS inner product on normalized = cosine)
faiss.normalize_L2(embeddings_float32)

# Build FAISS index
d = embeddings_float32.shape[1]  # Dimension
print(f"\nBuilding FAISS index: {len(nodes)} vectors, dim={d}")

# Method 1: Flat (exact) index with inner product (= cosine on normalized)
index_flat = faiss.IndexFlatIP(d)
index_flat.add(embeddings_float32)

# Method 2: IVF index (approximate, for larger graphs)
nlist = 4  # Number of clusters (use more for larger graphs)
quantizer = faiss.IndexFlatIP(d)
index_ivf = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)
index_ivf.train(embeddings_float32)
index_ivf.add(embeddings_float32)
index_ivf.nprobe = 2  # Search 2 clusters

print(f"FAISS Flat index size: {index_flat.ntotal} vectors")
print(f"FAISS IVF index size: {index_ivf.ntotal} vectors")

# Query: find k nearest neighbors for a node
k = 6  # k=6 because first result is the query itself
query_nodes = [0, 33, 16]

print(f"\n{'='*50}")
print(f"Nearest Neighbor Search Results (k={k-1}):")
print(f"{'='*50}")

for q_node in query_nodes:
    q_idx = nodes.index(q_node)
    query_vec = embeddings_float32[q_idx:q_idx+1]
    
    # Exact search
    D_exact, I_exact = index_flat.search(query_vec, k)
    
    # Approximate search
    D_approx, I_approx = index_ivf.search(query_vec, k)
    
    print(f"\nQuery Node: {q_node} (Community: {community_map[q_node]})")
    print(f"  Exact (Flat) Neighbors:")
    for i in range(1, k):  # Skip first (self)
        neighbor = nodes[I_exact[0][i]]
        sim = D_exact[0][i]
        print(f"    Node {neighbor:2d} (cosine_sim={sim:.4f}, community={community_map[neighbor]})")
    
    print(f"  Approx (IVF) Neighbors:")
    for i in range(1, k):
        neighbor = nodes[I_approx[0][i]]
        sim = D_approx[0][i]
        print(f"    Node {neighbor:2d} (cosine_sim={sim:.4f}, community={community_map[neighbor]})")

# Benchmark
import time
n_queries = 100
query_vecs = embeddings_float32[np.random.choice(len(nodes), n_queries)]

start = time.time()
index_flat.search(query_vecs, k)
time_flat = time.time() - start

start = time.time()
index_ivf.search(query_vecs, k)
time_ivf = time.time() - start

print(f"\n\nBenchmark ({n_queries} queries):")
print(f"  Exact (Flat): {time_flat*1000:.2f} ms")
print(f"  Approx (IVF): {time_ivf*1000:.2f} ms")
print(f"  Speedup: {time_flat/time_ivf:.1f}x")

In [0]:
# ============================================================
# TASK 6: GRAPH VISUALIZATION
# ============================================================
# Project high-dimensional embeddings into 2D/3D for visual
# inspection of graph structure and communities.
#
# Dimensionality reduction methods:
#   PCA:  Linear projection maximizing variance
#         X_2d = X · W where W = top-2 eigenvectors of XᵀX
#   
#   t-SNE: Non-linear, preserves local neighborhood structure
#         Minimizes KL(P||Q) where P=pairwise similarities in
#         high-D and Q=pairwise similarities in low-D
#         P_ij = exp(-||x_i-x_j||²/2σ²) / Σ exp(-||x_k-x_l||²/2σ²)
#         Q_ij = (1 + ||y_i-y_j||²)⁻¹ / Σ (1 + ||y_k-y_l||²)⁻¹
#
#   UMAP: Preserves both local and global structure
#         Uses fuzzy topological representation
# ============================================================

from sklearn.manifold import TSNE
import umap

print("=" * 60)
print("TASK 6: GRAPH VISUALIZATION")
print("=" * 60)

nodes, emb_dw = dw.get_all_embeddings()
_, emb_n2v_h = n2v_homophily.get_all_embeddings()
_, emb_n2v_s = n2v_structural.get_all_embeddings()
true_labels = np.array([community_map[n] for n in nodes])

# Apply dimensionality reduction
print("\nApplying dimensionality reduction...")

# PCA
pca = PCA(n_components=2, random_state=42)
emb_pca = pca.fit_transform(emb_n2v_h)
print(f"  PCA: explained variance = {pca.explained_variance_ratio_.sum():.4f}")

# t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=10, n_iter=1000)
emb_tsne = tsne.fit_transform(emb_n2v_h)
print(f"  t-SNE: KL divergence = {tsne.kl_divergence_:.4f}")

# UMAP
reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=10, min_dist=0.3)
emb_umap = reducer.fit_transform(emb_n2v_h)
print(f"  UMAP: complete")

# Create comprehensive visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Row 1: Different reduction methods on Node2Vec (homophily)
for ax, data, title in zip(
    axes[0], 
    [emb_pca, emb_tsne, emb_umap],
    ['PCA', 't-SNE', 'UMAP']
):
    scatter = ax.scatter(data[:, 0], data[:, 1], c=true_labels, 
                        cmap='Set1', s=150, edgecolors='black', 
                        linewidths=0.5, alpha=0.8)
    for i, node in enumerate(nodes):
        ax.annotate(str(node), (data[i, 0], data[i, 1]), 
                   fontsize=7, ha='center', va='center')
    ax.set_title(f'Node2Vec (Homophily) - {title}', fontsize=12)
    ax.set_xlabel(f'{title} Dim 1')
    ax.set_ylabel(f'{title} Dim 2')

# Row 2: Compare DeepWalk vs Node2Vec variants using t-SNE
for ax, emb, title in zip(
    axes[1],
    [emb_dw, emb_n2v_h, emb_n2v_s],
    ['DeepWalk', 'Node2Vec (q=0.5, Homophily)', 'Node2Vec (q=2, Structural)']
):
    tsne_tmp = TSNE(n_components=2, random_state=42, perplexity=10)
    data = tsne_tmp.fit_transform(emb)
    ax.scatter(data[:, 0], data[:, 1], c=true_labels, 
              cmap='Set1', s=150, edgecolors='black', 
              linewidths=0.5, alpha=0.8)
    for i, node in enumerate(nodes):
        ax.annotate(str(node), (data[i, 0], data[i, 1]), 
                   fontsize=7, ha='center', va='center')
    ax.set_title(f'{title} - t-SNE', fontsize=11)

plt.suptitle('Graph Embedding Visualizations', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n✔ Notice how Node2Vec (homophily, low q) separates communities more cleanly!")

In [0]:
# ============================================================
# TASK 7: FEATURE ENGINEERING FOR ML MODELS
# ============================================================
# Use node embeddings as additional features alongside
# traditional graph features for ML models.
# Use case: enriching feature sets for XGBoost, LightGBM, etc.
#
# Combined feature vector for node v:
#   x(v) = [Φ(v) || f_degree(v) || f_centrality(v) || ...]
#
# This combines:
#   - Structural features: degree, centrality, clustering coeff
#   - Embedding features: learned representations
#   - Interaction features: products between structural and embedding
# ============================================================

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score

print("=" * 60)
print("TASK 7: FEATURE ENGINEERING FOR ML")
print("=" * 60)

# Compute traditional graph features
nodes_sorted = sorted(G.nodes())

# Structural features
degree = np.array([G.degree(n) for n in nodes_sorted]).reshape(-1, 1)
betweenness = np.array([nx.betweenness_centrality(G)[n] for n in nodes_sorted]).reshape(-1, 1)
closeness = np.array([nx.closeness_centrality(G)[n] for n in nodes_sorted]).reshape(-1, 1)
clustering = np.array([nx.clustering(G)[n] for n in nodes_sorted]).reshape(-1, 1)
pagerank = np.array([nx.pagerank(G)[n] for n in nodes_sorted]).reshape(-1, 1)

# Stack traditional features
traditional_features = np.hstack([degree, betweenness, closeness, clustering, pagerank])
feature_names_trad = ['degree', 'betweenness', 'closeness', 'clustering', 'pagerank']

# Get embedding features
_, emb_features = n2v_homophily.get_all_embeddings()

# Combined features
combined_features = np.hstack([traditional_features, emb_features])

labels = np.array([community_map[n] for n in nodes_sorted])

# Compare feature sets using cross-validation
print("\nCross-validated classification accuracy (5-fold):")
print(f"{'Feature Set':<35} {'Accuracy':>10} {'F1':>10}")
print("-" * 60)

feature_sets = {
    'Traditional Only (5 features)': traditional_features,
    'Embedding Only (64 features)': emb_features,
    'Combined (69 features)': combined_features,
}

for name, features in feature_sets.items():
    clf = GradientBoostingClassifier(n_estimators=100, random_state=42, max_depth=3)
    cv_acc = cross_val_score(clf, features, labels, cv=5, scoring='accuracy')
    cv_f1 = cross_val_score(clf, features, labels, cv=5, scoring='f1_weighted')
    print(f"{name:<35} {cv_acc.mean():>10.4f} {cv_f1.mean():>10.4f}")

# Feature importance analysis
print("\nTop-10 Feature Importances (Combined model):")
clf_combined = GradientBoostingClassifier(n_estimators=100, random_state=42, max_depth=3)
clf_combined.fit(combined_features, labels)

# Create feature importance mapping
all_feature_names = feature_names_trad + [f'emb_{i}' for i in range(emb_features.shape[1])]
importances = clf_combined.feature_importances_
top_indices = np.argsort(importances)[::-1][:10]

for i, idx in enumerate(top_indices):
    print(f"  {i+1}. {all_feature_names[idx]:<15} importance={importances[idx]:.4f}")

In [0]:
# ============================================================
# TASK 8: ANOMALY DETECTION
# ============================================================
# Detect nodes whose embeddings deviate significantly from
# their neighbors or from expected clusters.
# Use case: fraud detection, network intrusion, bot detection
#
# Methods:
#   1. Local Outlier Factor (LOF): Compares local density
#      LOF(v) = avg_{u∈N_k(v)} [lrd(u) / lrd(v)]
#      where lrd(v) = local reachability density
#
#   2. Isolation Forest: Anomalies are easier to isolate
#      Score = 2^(-E[h(x)] / c(n))  where h(x) = path length
#
#   3. Embedding deviation: Compare node embedding to
#      average of its neighbors' embeddings:
#      anomaly_score(v) = ||Φ(v) - (1/|N(v)|)Σ_{u∈N(v)} Φ(u)||₂
# ============================================================

from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

print("=" * 60)
print("TASK 8: ANOMALY DETECTION")
print("=" * 60)

nodes, embeddings = n2v_homophily.get_all_embeddings()

# Method 1: Isolation Forest
iso_forest = IsolationForest(contamination=0.15, random_state=42)
iso_labels = iso_forest.fit_predict(embeddings)
iso_scores = iso_forest.score_samples(embeddings)

# Method 2: Local Outlier Factor
lof = LocalOutlierFactor(n_neighbors=5, contamination=0.15)
lof_labels = lof.fit_predict(embeddings)
lof_scores = lof.negative_outlier_factor_

# Method 3: Embedding deviation from neighborhood
def compute_neighborhood_deviation(graph, model, nodes):
    """Compute how much each node deviates from its neighbors' mean."""
    deviations = []
    for node in nodes:
        node_emb = model.get_embedding(node)
        neighbors = list(graph.neighbors(node))
        if len(neighbors) > 0:
            neighbor_embs = np.array([model.get_embedding(n) for n in neighbors])
            mean_neighbor = neighbor_embs.mean(axis=0)
            deviation = np.linalg.norm(node_emb - mean_neighbor)
        else:
            deviation = 0.0
        deviations.append(deviation)
    return np.array(deviations)

dev_scores = compute_neighborhood_deviation(G, n2v_homophily, nodes)

# Report anomalies
print("\nAnomaly Detection Results:")
print(f"\n{'Node':>5} {'IsoForest':>10} {'LOF':>10} {'Deviation':>10} {'Community':>10} {'Degree':>7}")
print("-" * 60)

# Get top anomalies by deviation score
top_anomalies = np.argsort(dev_scores)[::-1][:10]
for idx in top_anomalies:
    node = nodes[idx]
    print(f"{node:>5} {iso_scores[idx]:>10.4f} {lof_scores[idx]:>10.4f} "
          f"{dev_scores[idx]:>10.4f} {community_map[node]:>10} {G.degree(node):>7}")

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

tsne_anom = TSNE(n_components=2, random_state=42, perplexity=10)
X_2d = tsne_anom.fit_transform(embeddings)

for ax, scores, title in zip(
    axes,
    [iso_scores, lof_scores, dev_scores],
    ['Isolation Forest Score', 'LOF Score', 'Neighborhood Deviation']
):
    scatter = ax.scatter(X_2d[:, 0], X_2d[:, 1], c=scores, 
                        cmap='RdYlGn', s=150, edgecolors='black', linewidths=0.5)
    for i, node in enumerate(nodes):
        ax.annotate(str(node), (X_2d[i, 0], X_2d[i, 1]), fontsize=7, ha='center')
    ax.set_title(title, fontsize=11)
    plt.colorbar(scatter, ax=ax)

plt.suptitle('Anomaly Detection in Embedding Space', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n✔ Nodes at community boundaries (e.g., 2, 8, 13, 31) show highest deviation!")

## Task 9: Fraud Ring / Risk Cluster Discovery

### Description
Fraud ring discovery identifies tightly-connected groups of suspicious entities in a graph. Unlike individual anomaly detection, this focuses on **collusive behavior** — groups of nodes that act together and are structurally cohesive.

**Real-world applications:** Fraud ring detection in financial networks, collusion detection in marketplaces, money laundering network discovery, organized crime network analysis.

### Input Data
- **Graph:** Augmented with a synthetic fraud ring (tightly-connected subgraph with sparse external links)
- **Embeddings:** Node2Vec vectors trained on the augmented graph
- **No fraud labels** needed (unsupervised clustering approach)

### Mathematical Formulation

**Risk scoring for a cluster $C$:**
$$\text{risk}(C) = \alpha \cdot \text{density}(C) + \beta \cdot \text{isolation}(C)$$

where:
$$\text{density}(C) = \frac{|\text{edges within } C|}{\binom{|C|}{2}} = \frac{2|E_C|}{|C|(|C|-1)}$$
$$\text{isolation}(C) = 1 - \frac{|\text{edges between } C \text{ and rest}|}{|\text{all edges touching } C|}$$

**Intuition:** Fraud rings exhibit:
- High internal density (members transact heavily with each other)
- High isolation (few connections to legitimate network)
- Tight clustering in embedding space (shared structural context)

### Expected Outcome
- Automatic discovery of the injected fraud ring via HDBSCAN
- Risk scores per cluster (density + isolation)
- Precision/recall of fraud node recovery

## Task 10: Search and Ranking

### Description
Search and ranking improves retrieval by representing entities as graph embeddings and computing vector similarity. Given a query node, all candidates are ranked by their embedding proximity.

**Real-world applications:** User search, product ranking, document retrieval, personalized search results, entity matching.

### Input Data
- **Query:** A node $q$ with embedding $\Phi(q)$
- **Candidates:** All other nodes with their embeddings
- **Relevance ground truth:** Nodes in the same community as $q$ are considered "relevant"

### Mathematical Formulation

**Ranking function:**
$$\text{rank}(c) = \text{sim}(\Phi(q), \Phi(c)), \quad c \in \text{Candidates}$$

**Evaluation metrics:**

*Mean Reciprocal Rank (MRR):*
$$\text{MRR} = \frac{1}{|Q|} \sum_{i=1}^{|Q|} \frac{1}{\text{rank}_i}$$

*Normalized Discounted Cumulative Gain (NDCG@k):*
$$\text{DCG@}k = \sum_{i=1}^{k} \frac{\text{rel}_i}{\log_2(i+1)}, \quad \text{NDCG@}k = \frac{\text{DCG@}k}{\text{IDCG@}k}$$

*Precision@K:*
$$\text{P@}k = \frac{|\{\text{relevant items in top-}k\}|}{k}$$

### Expected Outcome
- Ranked results for multiple query nodes with similarity scores
- MRR, NDCG@10, and Precision@K metrics
- Demonstration that embedding-based ranking retrieves community-relevant nodes

## Task 11: Cold-Start Recommendations

### Description
The cold-start problem occurs when a new entity has no historical interaction data. Graph embeddings solve this: even without interaction history, a new node inherits structural context from its neighbors in the graph.

**Real-world applications:** New user onboarding, new product launches, new employee recommendations, bootstrapping content recommendations.

### Input Data
- **New node:** $v_{\text{new}}$ with known graph connections but no pre-trained embedding
- **Neighbors:** $\mathcal{N}(v_{\text{new}})$ — nodes the new entity is connected to
- **Pre-trained embeddings:** $\Phi(u)$ for all existing nodes $u$

### Mathematical Formulation

**Embedding approximation methods:**

*Mean aggregation:*
$$\hat{\Phi}(v_{\text{new}}) = \frac{1}{|\mathcal{N}(v_{\text{new}})|} \sum_{u \in \mathcal{N}(v_{\text{new}})} \Phi(u)$$

*Degree-weighted aggregation:*
$$\hat{\Phi}(v_{\text{new}}) = \sum_{u \in \mathcal{N}(v_{\text{new}})} \frac{\deg(u)}{\sum_{w \in \mathcal{N}} \deg(w)} \cdot \Phi(u)$$

*Max-pooling:*
$$\hat{\Phi}(v_{\text{new}})_j = \max_{u \in \mathcal{N}(v_{\text{new}})} \Phi(u)_j \quad \forall j \in [1, d]$$

**Quality measure:**
$$\text{quality} = \text{cosine}(\hat{\Phi}(v_{\text{new}}), \Phi_{\text{true}}(v_{\text{new}}))$$

### Expected Outcome
- Approximation quality (cosine similarity to true embedding)
- Cold-start recommendations for held-out nodes
- Same-community hit rate showing the approach works even without training

## Task 12: Node Retrieval / Entity Resolution

### Description
Entity resolution identifies duplicate or semantically equivalent entities in a graph. Nodes that are structurally similar (similar neighborhoods, roles) will have similar embeddings, enabling duplicate detection via similarity thresholding.

**Real-world applications:** Duplicate customer account detection, merchant matching, author disambiguation, record linkage across databases.

### Input Data
- **Embeddings:** $\Phi(v) \in \mathbb{R}^d$ for all nodes
- **Threshold:** $\tau$ — similarity cutoff for declaring a match
- **Structural features:** Jaccard similarity of neighborhoods for validation

### Mathematical Formulation

**Entity resolution rule:**
$$\text{match}(u, v) = \mathbb{1}\left[\text{sim}(\Phi(u), \Phi(v)) > \tau\right]$$

**Combined scoring:**
$$\text{score}(u, v) = \alpha \cdot \text{cosine}(\Phi(u), \Phi(v)) + (1-\alpha) \cdot \text{Jaccard}(\mathcal{N}(u), \mathcal{N}(v))$$

where:
$$\text{Jaccard}(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

**Blocking strategy** (for scalability):
- Only compare nodes with similar degrees: $|\deg(u) - \deg(v)| \leq \delta$
- Reduces $O(n^2)$ comparisons to $O(n \cdot b)$ where $b$ = block size

### Expected Outcome
- Candidate entity pairs ranked by combined score
- Validation: matched pairs tend to share community membership
- Demonstration of blocking for computational efficiency

## Task 13: Graph-Based Customer Segmentation

### Description
Traditional customer segmentation uses demographic features (age, income, location). Graph-based segmentation additionally leverages **relational structure** — who interacts with whom — capturing behavioral patterns invisible to feature-only approaches.

**Real-world applications:** Marketing campaign targeting, personalized pricing, churn prediction cohorts, loyalty program design.

### Input Data
- **Demographic features:** $f_{\text{demo}}(v) \in \mathbb{R}^p$ (age, income, activity, tenure)
- **Graph embeddings:** $\Phi(v) \in \mathbb{R}^d$ (structural/relational patterns)
- **Combined features:** $x(v) = [f_{\text{demo}}(v) \| \Phi(v)] \in \mathbb{R}^{p+d}$

### Mathematical Formulation

**Agglomerative (Hierarchical) Clustering with Ward's linkage:**
$$d_{\text{Ward}}(C_i, C_j) = \frac{|C_i| \cdot |C_j|}{|C_i| + |C_j|} \|\mu_i - \mu_j\|^2$$

Ward's method merges clusters that cause the minimum increase in total within-cluster variance.

**Three segmentation approaches compared:**
1. Demographics only: $x = f_{\text{demo}}(v)$
2. Embeddings only: $x = \Phi(v)$
3. Combined: $x = [f_{\text{demo}}(v) \| \Phi(v)]$

**Hypothesis:** Combined segmentation captures both *who the customer is* (demographics) and *how the customer behaves* (graph structure).

### Expected Outcome
- NMI/ARI comparison across three approaches
- Segment profiles with demographic and structural characteristics
- Visual demonstration that combined features produce better-defined segments

## Task 14: Knowledge Graph Completion

### Description
Knowledge Graph Completion (KGC) predicts missing relationships (triples) in a knowledge graph. Using pre-trained node embeddings combined with learned relation vectors, we can score candidate triples and rank missing links.

**Real-world applications:** Filling gaps in ontologies, drug-target prediction, recommendation via relation prediction, question answering over knowledge bases.

### Input Data
- **Knowledge Graph:** $\mathcal{G} = (E, R, T)$ with entities $E$, relations $R$, triples $T = \{(h, r, t)\}$
- **Node embeddings:** $\Phi(e) \in \mathbb{R}^d$ from Node2Vec
- **Relation embeddings:** Learned as average translation vectors

### Mathematical Formulation

**TransE model** (Bordes et al., 2013):
$$\text{Score}(h, r, t) = -\|\Phi(h) + \Phi(r) - \Phi(t)\|_2$$

Intuition: For a correct triple, $\Phi(h) + \Phi(r) \approx \Phi(t)$ (the relation acts as a translation).

**Learning relation embeddings from data:**
$$\Phi(r) = \frac{1}{|T_r|} \sum_{(h, r, t) \in T_r} \left[\Phi(t) - \Phi(h)\right]$$

**Other scoring functions (for reference):**
- DistMult: $\text{Score}(h, r, t) = \sum_i \Phi(h)_i \cdot \Phi(r)_i \cdot \Phi(t)_i$
- ComplEx: $\text{Score}(h, r, t) = \text{Re}\left(\sum_i \Phi(h)_i \cdot \Phi(r)_i \cdot \overline{\Phi(t)}_i\right)$

**Evaluation:** For query $(h, r, ?)$, rank all entities by score and measure Hits@K.

### Expected Outcome
- Relation embeddings learned as average translations
- Tail entity prediction with ranked candidates
- Hits@10 metric for knowledge graph completion

## Task 15: Transfer Learning for Graph Tasks

### Description
Transfer learning uses pre-trained node embeddings as initialization for downstream neural networks, rather than starting from random weights. This provides a **structural prior** that accelerates learning and improves performance, especially with limited labeled data.

**Real-world applications:** Few-shot node classification, pre-training for GNNs, cross-domain transfer, label-efficient learning.

### Input Data
- **Pre-trained embeddings:** $\Phi_{\text{pretrained}} \in \mathbb{R}^{n \times d}$ (from Node2Vec)
- **Random baseline:** $\Phi_{\text{random}} \sim \mathcal{N}(0, 0.01^2 I)$
- **Task labels:** Community labels for classification
- **Varying label budgets:** 10%, 20%, ..., 100% of training data

### Mathematical Formulation

**Pre-trained initialization:**
$$\Phi_0 = \Phi_{\text{pretrained}} \quad \text{(encodes graph topology)}$$

**Fine-tuning with regularization:**
$$\Phi^* = \arg\min_{\Phi} \left[ \mathcal{L}_{\text{task}}(\Phi) + \lambda \|\Phi - \Phi_0\|_F^2 \right]$$

The regularization term prevents the fine-tuned embeddings from drifting too far from the pre-trained structure.

**Why pre-training helps:**
- Random init: optimizer must learn both **structural representation** + **task decision boundary**
- Pre-trained init: structure already encoded, optimizer only learns the **decision boundary**
- Formally: pre-training reduces the effective hypothesis space, improving generalization

**Label efficiency:** With only $k$ labeled samples:
$$\text{Acc}_{\text{pretrained}}(k) \gg \text{Acc}_{\text{random}}(k) \quad \text{for small } k$$

### Expected Outcome
- Learning curves: pre-trained converges faster (lower loss earlier)
- Label efficiency study: pre-trained achieves higher accuracy with fewer labels
- Quantified improvement in accuracy (percentage points gained)

In [0]:
# ============================================================
# TASK 9: FRAUD RING / RISK CLUSTER DISCOVERY
# ============================================================
# Identify clusters of interconnected suspicious nodes.
# Use case: fraud rings, collusion detection, money laundering
#
# Approach:
#   1. Generate node embeddings
#   2. Cluster embeddings using HDBSCAN (density-based)
#   3. Analyze cluster properties to identify risky groups
#   4. Score clusters by internal connectivity & anomaly
#
# Risk Score for cluster C:
#   risk(C) = α·density(C) + β·avg_anomaly(C) + γ·isolation(C)
#   where:
#     density = |edges within C| / (|C| choose 2)
#     avg_anomaly = mean anomaly score of nodes in C
#     isolation = 1 - |edges between C and rest| / |all edges of C|
# ============================================================

print("=" * 60)
print("TASK 9: FRAUD RING / RISK CLUSTER DISCOVERY")
print("=" * 60)

# Simulate a scenario: add suspicious nodes forming a ring
G_fraud = G.copy()

# Add a "fraud ring" — tightly connected group with few external links
fraud_nodes = list(range(34, 40))  # 6 new nodes
for node in fraud_nodes:
    G_fraud.add_node(node)
    community_map[node] = 2  # New "fraud" community

# Connect fraud nodes in a ring + internal edges
for i in range(len(fraud_nodes)):
    G_fraud.add_edge(fraud_nodes[i], fraud_nodes[(i+1) % len(fraud_nodes)])
    if i + 2 < len(fraud_nodes):
        G_fraud.add_edge(fraud_nodes[i], fraud_nodes[i+2])

# Add few connections to main graph (sparse external links)
G_fraud.add_edge(34, 0)
G_fraud.add_edge(35, 33)

print(f"Augmented graph: {G_fraud.number_of_nodes()} nodes, {G_fraud.number_of_edges()} edges")
print(f"Fraud ring nodes: {fraud_nodes}")

# Train Node2Vec on augmented graph
n2v_fraud = Node2Vec(
    graph=G_fraud, walk_length=40, num_walks=80,
    embedding_dim=64, window_size=5, p=1.0, q=0.5, seed=42
)
n2v_fraud.fit()

# Get embeddings and cluster
nodes_f, emb_f = n2v_fraud.get_all_embeddings()
scaler_f = StandardScaler()
emb_f_scaled = scaler_f.fit_transform(emb_f)

# HDBSCAN clustering
hdb_fraud = hdbscan.HDBSCAN(min_cluster_size=4, min_samples=2)
cluster_labels = hdb_fraud.fit_predict(emb_f_scaled)

# Analyze clusters
print(f"\nClusters found: {len(set(cluster_labels) - {-1})}")
print(f"Noise points: {sum(cluster_labels == -1)}")

def compute_cluster_risk(graph, nodes_list, cluster_labels, cluster_id):
    """Compute risk metrics for a cluster."""
    cluster_nodes = [nodes_list[i] for i in range(len(nodes_list)) if cluster_labels[i] == cluster_id]
    n = len(cluster_nodes)
    if n < 2:
        return {'nodes': cluster_nodes, 'size': n, 'density': 0, 'isolation': 0}
    
    # Internal edges
    cluster_set = set(cluster_nodes)
    internal_edges = sum(1 for u, v in graph.edges() 
                        if u in cluster_set and v in cluster_set)
    max_edges = n * (n-1) / 2
    density = internal_edges / max_edges if max_edges > 0 else 0
    
    # External edges
    external_edges = sum(1 for u in cluster_nodes 
                        for v in graph.neighbors(u) if v not in cluster_set)
    total_edges = internal_edges * 2 + external_edges
    isolation = 1 - (external_edges / total_edges) if total_edges > 0 else 0
    
    return {
        'nodes': cluster_nodes, 'size': n, 
        'density': density, 'isolation': isolation,
        'risk_score': 0.5 * density + 0.5 * isolation
    }

print(f"\n{'Cluster':>8} {'Size':>5} {'Density':>8} {'Isolation':>10} {'Risk':>6} {'Nodes'}")
print("-" * 70)

for cid in sorted(set(cluster_labels) - {-1}):
    metrics = compute_cluster_risk(G_fraud, nodes_f, cluster_labels, cid)
    nodes_str = str(metrics['nodes'][:8]) + ('...' if len(metrics['nodes']) > 8 else '')
    print(f"{cid:>8} {metrics['size']:>5} {metrics['density']:>8.4f} "
          f"{metrics['isolation']:>10.4f} {metrics['risk_score']:>6.3f} {nodes_str}")

# Check if fraud ring was detected
fraud_set = set(fraud_nodes)
for cid in set(cluster_labels) - {-1}:
    cluster_nodes = set(nodes_f[i] for i in range(len(nodes_f)) if cluster_labels[i] == cid)
    overlap = cluster_nodes & fraud_set
    if len(overlap) > 0:
        recall = len(overlap) / len(fraud_set)
        precision = len(overlap) / len(cluster_nodes)
        print(f"\n⚠️  Cluster {cid} contains {len(overlap)}/{len(fraud_set)} fraud nodes!")
        print(f"   Precision: {precision:.2%}, Recall: {recall:.2%}")

In [0]:
# ============================================================
# TASK 10: SEARCH AND RANKING
# ============================================================
# Improve retrieval and ranking by computing vector similarity
# between queries and candidates in embedding space.
# Use case: user search, product ranking, document retrieval
#
# Mathematical formulation:
#   Given query node q and candidate set C:
#   rank(c) = sim(Φ(q), Φ(c)) for c ∈ C
#
#   Ranking metrics:
#   - MRR (Mean Reciprocal Rank) = (1/|Q|) Σ 1/rank_i
#   - NDCG = DCG/IDCG where DCG = Σ rel_i / log₂(i+1)
#   - Precision@K = |relevant in top-K| / K
# ============================================================

print("=" * 60)
print("TASK 10: SEARCH AND RANKING")
print("=" * 60)

def embedding_search_rank(model, query_node, candidates, true_relevant):
    """
    Rank candidates by similarity to query in embedding space.
    Returns ranked list and evaluation metrics.
    """
    query_emb = model.get_embedding(query_node)
    
    scores = []
    for c in candidates:
        c_emb = model.get_embedding(c)
        sim = np.dot(query_emb, c_emb) / (
            np.linalg.norm(query_emb) * np.linalg.norm(c_emb) + 1e-8
        )
        scores.append((c, sim))
    
    # Sort by similarity (descending)
    ranked = sorted(scores, key=lambda x: x[1], reverse=True)
    
    # Compute metrics
    relevant_set = set(true_relevant)
    
    # MRR: rank of first relevant item
    mrr = 0.0
    for i, (node, _) in enumerate(ranked):
        if node in relevant_set:
            mrr = 1.0 / (i + 1)
            break
    
    # Precision@K
    k_values = [3, 5, 10]
    precisions = {}
    for k in k_values:
        top_k = set(node for node, _ in ranked[:k])
        precisions[k] = len(top_k & relevant_set) / k
    
    # NDCG@10
    dcg = 0.0
    for i, (node, _) in enumerate(ranked[:10]):
        rel = 1.0 if node in relevant_set else 0.0
        dcg += rel / np.log2(i + 2)
    
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(relevant_set), 10)))
    ndcg = dcg / idcg if idcg > 0 else 0.0
    
    return ranked, {'mrr': mrr, 'ndcg@10': ndcg, **{f'p@{k}': v for k, v in precisions.items()}}

# Demo: Search for nodes similar to query
# "Relevant" = same community members
print("\nSearch & Ranking Demo:")
print("Query: Find nodes most similar to the query node")
print("Relevance: Same community as query\n")

all_metrics = []
for query_node in [0, 33, 2, 13, 27]:
    query_community = community_map[query_node]
    candidates = [n for n in G.nodes() if n != query_node]
    true_relevant = [n for n in G.nodes() if community_map[n] == query_community and n != query_node]
    
    ranked, metrics = embedding_search_rank(
        n2v_homophily, query_node, candidates, true_relevant
    )
    all_metrics.append(metrics)
    
    print(f"Query Node {query_node} (Community {query_community}):")
    print(f"  Top-5 results: {[(n, f'{s:.3f}') for n, s in ranked[:5]]}")
    print(f"  MRR={metrics['mrr']:.3f}, NDCG@10={metrics['ndcg@10']:.3f}, "
          f"P@3={metrics['p@3']:.3f}, P@5={metrics['p@5']:.3f}")
    print()

# Average metrics
print("\nAverage Ranking Metrics:")
avg_metrics = {k: np.mean([m[k] for m in all_metrics]) for k in all_metrics[0].keys()}
for k, v in avg_metrics.items():
    print(f"  {k.upper()}: {v:.4f}")

In [0]:
# ============================================================
# TASK 11: COLD-START RECOMMENDATIONS
# ============================================================
# Recommend items/users even when historical interaction data
# is sparse or unavailable (the "cold start" problem).
# Use case: new user onboarding, new product launch
#
# Key Insight:
#   Even with zero interaction history, a new node connected
#   to the graph inherits structural context from its neighbors.
#
# Approach for cold-start node v_new:
#   1. Connect v_new to known neighbors in the graph
#   2. Approximate embedding: Φ(v_new) ≈ (1/|N|) Σ_{u∈N(v_new)} Φ(u)
#   3. Use approximate embedding for recommendations
#
# Alternative (inductive):
#   Φ(v_new) = f(features(v_new), Φ(neighbors(v_new)))
# ============================================================

print("=" * 60)
print("TASK 11: COLD-START RECOMMENDATIONS")
print("=" * 60)

def cold_start_embedding(graph, model, new_node_neighbors, method='mean'):
    """
    Approximate embedding for a new node based on its neighbors.
    
    Methods:
    - mean: average of neighbors' embeddings
    - weighted: degree-weighted average
    - max_pool: element-wise maximum
    """
    neighbor_embs = np.array([model.get_embedding(n) for n in new_node_neighbors])
    
    if method == 'mean':
        return neighbor_embs.mean(axis=0)
    elif method == 'weighted':
        weights = np.array([graph.degree(n) for n in new_node_neighbors], dtype=float)
        weights /= weights.sum()
        return np.average(neighbor_embs, axis=0, weights=weights)
    elif method == 'max_pool':
        return neighbor_embs.max(axis=0)
    else:
        raise ValueError(f"Unknown method: {method}")

# Simulate cold-start: "remove" a node and try to reconstruct
print("\nSimulating Cold-Start Scenario:")
print("Remove node from training, reconnect, approximate embedding.\n")

test_nodes = [2, 13, 27, 5]  # Nodes to test cold-start on

for test_node in test_nodes:
    neighbors = list(G.neighbors(test_node))
    true_community = community_map[test_node]
    
    # Approximate embedding using different methods
    for method in ['mean', 'weighted', 'max_pool']:
        approx_emb = cold_start_embedding(G, n2v_homophily, neighbors, method)
        true_emb = n2v_homophily.get_embedding(test_node)
        
        # How close is the approximation?
        cosine_sim = np.dot(approx_emb, true_emb) / (
            np.linalg.norm(approx_emb) * np.linalg.norm(true_emb) + 1e-8
        )
        
        # Can we still find the right community?
        # Find nearest existing node to the approximated embedding
        best_sim = -1
        best_node = -1
        for n in G.nodes():
            if n == test_node:
                continue
            n_emb = n2v_homophily.get_embedding(n)
            sim = np.dot(approx_emb, n_emb) / (
                np.linalg.norm(approx_emb) * np.linalg.norm(n_emb) + 1e-8
            )
            if sim > best_sim:
                best_sim = sim
                best_node = n
        
        if method == 'mean':  # Print once per node
            print(f"Node {test_node} (Community {true_community}, Degree {len(neighbors)}):")
    
    # Show results for mean method
    approx_emb = cold_start_embedding(G, n2v_homophily, neighbors, 'mean')
    true_emb = n2v_homophily.get_embedding(test_node)
    cosine_sim = np.dot(approx_emb, true_emb) / (
        np.linalg.norm(approx_emb) * np.linalg.norm(true_emb) + 1e-8
    )
    
    # Recommend top-3 from approximate embedding
    sims = []
    for n in G.nodes():
        if n == test_node:
            continue
        n_emb = n2v_homophily.get_embedding(n)
        s = np.dot(approx_emb, n_emb) / (np.linalg.norm(approx_emb) * np.linalg.norm(n_emb) + 1e-8)
        sims.append((n, s))
    sims.sort(key=lambda x: x[1], reverse=True)
    
    recs = sims[:5]
    same_comm = sum(1 for n, _ in recs if community_map[n] == true_community)
    print(f"  Approximation quality (cosine to true): {cosine_sim:.4f}")
    print(f"  Cold-start recs: {[(n, f'{s:.3f}') for n, s in recs]}")
    print(f"  Same-community hits: {same_comm}/5")
    print()

In [0]:
# ============================================================
# TASK 12: NODE RETRIEVAL / ENTITY RESOLUTION
# ============================================================
# Find duplicate or semantically similar entities in graphs.
# Use case: duplicate customer accounts, merchant matching
#
# Mathematical formulation:
#   Entity resolution as a threshold-based clustering:
#   Two nodes u, v are duplicates if:
#     sim(Φ(u), Φ(v)) > τ  (threshold)
#   AND structural_sim(u, v) > τ_struct
#
# Structural similarity metrics:
#   - Jaccard: |N(u) ∩ N(v)| / |N(u) ∪ N(v)|
#   - Adamic-Adar: Σ_{w∈N(u)∩N(v)} 1/log|N(w)|
#   - Common Neighbors: |N(u) ∩ N(v)|
# ============================================================

print("=" * 60)
print("TASK 12: NODE RETRIEVAL / ENTITY RESOLUTION")
print("=" * 60)

def find_similar_entities(model, graph, threshold=0.8, top_k=10):
    """
    Find pairs of nodes with high embedding similarity.
    These are candidate duplicates/similar entities.
    """
    nodes = sorted(graph.nodes())
    candidates = []
    
    for i in range(len(nodes)):
        emb_i = model.get_embedding(nodes[i])
        for j in range(i+1, len(nodes)):
            emb_j = model.get_embedding(nodes[j])
            
            # Cosine similarity
            cos_sim = np.dot(emb_i, emb_j) / (
                np.linalg.norm(emb_i) * np.linalg.norm(emb_j) + 1e-8
            )
            
            if cos_sim > threshold:
                # Also compute structural similarity (Jaccard)
                neighbors_i = set(graph.neighbors(nodes[i]))
                neighbors_j = set(graph.neighbors(nodes[j]))
                jaccard = len(neighbors_i & neighbors_j) / len(neighbors_i | neighbors_j) if len(neighbors_i | neighbors_j) > 0 else 0
                
                candidates.append({
                    'node_a': nodes[i], 'node_b': nodes[j],
                    'embedding_sim': cos_sim, 'jaccard_sim': jaccard,
                    'combined_score': 0.7 * cos_sim + 0.3 * jaccard
                })
    
    # Sort by combined score
    candidates.sort(key=lambda x: x['combined_score'], reverse=True)
    return candidates[:top_k]

# Find similar entity pairs
print("\nTop candidate entity pairs (potential duplicates):")
print(f"{'Node A':>7} {'Node B':>7} {'Emb Sim':>8} {'Jaccard':>8} {'Combined':>9} {'Same Comm':>10}")
print("-" * 55)

candidates = find_similar_entities(n2v_homophily, G, threshold=0.7, top_k=15)
for c in candidates:
    same_comm = '✓' if community_map[c['node_a']] == community_map[c['node_b']] else '✗'
    print(f"{c['node_a']:>7} {c['node_b']:>7} {c['embedding_sim']:>8.4f} "
          f"{c['jaccard_sim']:>8.4f} {c['combined_score']:>9.4f} {same_comm:>10}")

# Entity resolution with blocking (for scalability)
print("\n\nEntity Resolution with Blocking Strategy:")
print("Block by: similar degree range (to reduce O(n²) comparisons)")

def blocked_entity_resolution(model, graph, degree_tolerance=3, sim_threshold=0.8):
    """Entity resolution with degree-based blocking."""
    nodes = sorted(graph.nodes())
    degrees = {n: graph.degree(n) for n in nodes}
    
    # Block: only compare nodes with similar degrees
    matches = []
    for i in range(len(nodes)):
        for j in range(i+1, len(nodes)):
            if abs(degrees[nodes[i]] - degrees[nodes[j]]) <= degree_tolerance:
                emb_i = model.get_embedding(nodes[i])
                emb_j = model.get_embedding(nodes[j])
                sim = np.dot(emb_i, emb_j) / (
                    np.linalg.norm(emb_i) * np.linalg.norm(emb_j) + 1e-8
                )
                if sim > sim_threshold:
                    matches.append((nodes[i], nodes[j], sim))
    
    return matches

matches = blocked_entity_resolution(n2v_homophily, G, degree_tolerance=2, sim_threshold=0.85)
print(f"\nFound {len(matches)} high-similarity pairs (threshold=0.85):")
for a, b, sim in matches[:10]:
    print(f"  ({a}, {b}): sim={sim:.4f}, degrees=({G.degree(a)}, {G.degree(b)})")

In [0]:
# ============================================================
# TASK 13: GRAPH-BASED CUSTOMER SEGMENTATION
# ============================================================
# Segment customers based on behavioral relationships (graph
# structure) rather than only demographic features.
# Use case: marketing, personalization, churn prediction
#
# Traditional segmentation uses features like:
#   age, income, location, purchase_amount
#
# Graph-based segmentation additionally captures:
#   - Who interacts with whom
#   - Influence propagation patterns
#   - Behavioral similarity through shared connections
#
# Combined approach:
#   x(v) = [Φ_graph(v) || features_demographic(v)]
#   Apply hierarchical clustering on combined vectors
# ============================================================

from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage

print("=" * 60)
print("TASK 13: GRAPH-BASED CUSTOMER SEGMENTATION")
print("=" * 60)

# Simulate demographic features for nodes
np.random.seed(42)
nodes_sorted = sorted(G.nodes())
n_nodes = len(nodes_sorted)

# Create synthetic demographic features
demographic_features = pd.DataFrame({
    'node': nodes_sorted,
    'age': np.random.normal(35, 10, n_nodes).astype(int),
    'income': np.random.normal(50000, 15000, n_nodes).astype(int),
    'activity_score': np.array([G.degree(n) for n in nodes_sorted]) + np.random.normal(0, 2, n_nodes),
    'tenure_months': np.random.randint(1, 60, n_nodes)
})

print("\nSynthetic Demographic Features (first 5):")
print(demographic_features.head().to_string())

# Get embedding features
_, emb_seg = n2v_homophily.get_all_embeddings()

# Three segmentation approaches
demo_features = StandardScaler().fit_transform(
    demographic_features[['age', 'income', 'activity_score', 'tenure_months']].values
)
emb_features_scaled = StandardScaler().fit_transform(emb_seg)
combined = np.hstack([demo_features, emb_features_scaled])

# Hierarchical clustering
n_segments = 4

segmentations = {}
for name, features in [('Demographics Only', demo_features), 
                        ('Graph Embeddings Only', emb_features_scaled),
                        ('Combined', combined)]:
    agg = AgglomerativeClustering(n_clusters=n_segments, linkage='ward')
    labels = agg.fit_predict(features)
    segmentations[name] = labels

# Evaluate segmentations against ground truth
print(f"\n\nSegmentation Quality (vs ground truth communities):")
print(f"{'Method':<25} {'NMI':>8} {'ARI':>8} {'Silhouette':>11}")
print("-" * 55)

true_labels = np.array([community_map[n] for n in nodes_sorted])
for name, labels in segmentations.items():
    nmi = normalized_mutual_info_score(true_labels, labels)
    ari = adjusted_rand_score(true_labels, labels)
    sil = silhouette_score(combined, labels)
    print(f"{name:<25} {nmi:>8.4f} {ari:>8.4f} {sil:>11.4f}")

# Segment profiling
print("\n\nSegment Profiles (Combined approach):")
combined_labels = segmentations['Combined']
for seg in range(n_segments):
    mask = combined_labels == seg
    seg_df = demographic_features[mask]
    seg_communities = true_labels[mask]
    print(f"\n  Segment {seg} ({mask.sum()} nodes):")
    print(f"    Avg Age: {seg_df['age'].mean():.0f}, Avg Income: ${seg_df['income'].mean():,.0f}")
    print(f"    Avg Degree: {seg_df['activity_score'].mean():.1f}")
    print(f"    Community distribution: {dict(zip(*np.unique(seg_communities, return_counts=True)))}")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
tsne_seg = TSNE(n_components=2, random_state=42, perplexity=8)
X_2d_seg = tsne_seg.fit_transform(combined)

for ax, (name, labels) in zip(axes, segmentations.items()):
    scatter = ax.scatter(X_2d_seg[:, 0], X_2d_seg[:, 1], c=labels, 
                        cmap='tab10', s=120, edgecolors='black', linewidths=0.5)
    ax.set_title(f'{name}\n({n_segments} segments)', fontsize=11)
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')

plt.suptitle('Customer Segmentation Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [0]:
# ============================================================
# TASK 14: KNOWLEDGE GRAPH COMPLETION
# ============================================================
# Predict missing relationships or entities in knowledge graphs.
# Use case: fill gaps in ontologies, recommend new connections
#
# Knowledge Graph: G = (E, R, T)
#   E = entities, R = relations, T = triples (h, r, t)
#   h = head entity, r = relation, t = tail entity
#
# Scoring functions for triple (h, r, t):
#   TransE: ||Φ(h) + Φ(r) - Φ(t)||₂  (translation model)
#   DistMult: Σ_i Φ(h)_i · Φ(r)_i · Φ(t)_i  (bilinear)
#   ComplEx: Re(Σ_i Φ(h)_i · Φ(r)_i · conj(Φ(t)_i))  (complex)
#
# Here we use node embeddings + relation-aware scoring
# ============================================================

print("=" * 60)
print("TASK 14: KNOWLEDGE GRAPH COMPLETION")
print("=" * 60)

# Create a simple knowledge graph from the Karate Club
# Relations: 'friend', 'same_club', 'bridge' (connects communities)
knowledge_triples = []

for u, v in G.edges():
    if community_map[u] == community_map[v]:
        knowledge_triples.append((u, 'same_club', v))
    else:
        knowledge_triples.append((u, 'bridge', v))
    knowledge_triples.append((u, 'friend', v))

print(f"Knowledge Graph:")
print(f"  Entities: {G.number_of_nodes()}")
print(f"  Relations: {len(set(r for _, r, _ in knowledge_triples))}")
print(f"  Triples: {len(knowledge_triples)}")
print(f"  Relation counts:")
for rel in set(r for _, r, _ in knowledge_triples):
    count = sum(1 for _, r, _ in knowledge_triples if r == rel)
    print(f"    {rel}: {count}")

# TransE-inspired scoring using pre-trained embeddings
# Score(h, r, t) = -||emb(h) + emb(r) - emb(t)||_2
# We learn relation embeddings as average translation vectors

def learn_relation_embeddings(triples, node_model, dim=64):
    """Learn relation embeddings as average translation: r ≈ t - h"""
    relation_vecs = defaultdict(list)
    
    for h, r, t in triples:
        h_emb = node_model.get_embedding(h)
        t_emb = node_model.get_embedding(t)
        translation = t_emb - h_emb  # r ≈ t - h
        relation_vecs[r].append(translation)
    
    relation_embeddings = {}
    for r, vecs in relation_vecs.items():
        relation_embeddings[r] = np.mean(vecs, axis=0)
    
    return relation_embeddings

relation_embs = learn_relation_embeddings(knowledge_triples, n2v_homophily)

def score_triple(h, r, t, node_model, relation_embs):
    """TransE score: -||h + r - t||_2 (higher is better)"""
    h_emb = node_model.get_embedding(h)
    t_emb = node_model.get_embedding(t)
    r_emb = relation_embs[r]
    return -np.linalg.norm(h_emb + r_emb - t_emb)

# Knowledge Graph Completion: predict missing triples
print("\n\nKG Completion — Predicting Missing Links:")
print("For each query (h, r, ?), rank all possible tail entities\n")

# Test: predict tails
test_queries = [(0, 'same_club'), (33, 'bridge'), (2, 'friend')]

for h, r in test_queries:
    scores = []
    for t in G.nodes():
        if t != h:
            s = score_triple(h, r, t, n2v_homophily, relation_embs)
            scores.append((t, s))
    
    scores.sort(key=lambda x: x[1], reverse=True)
    
    # Ground truth
    true_tails = set(t for hh, rr, t in knowledge_triples if hh == h and rr == r)
    
    print(f"Query: ({h}, {r}, ?)")
    print(f"  True tails: {sorted(true_tails)[:10]}")
    print(f"  Top-5 predictions: {[(t, f'{s:.3f}') for t, s in scores[:5]]}")
    
    # Hits@K
    top_10 = set(t for t, _ in scores[:10])
    hits_10 = len(top_10 & true_tails) / min(len(true_tails), 10)
    print(f"  Hits@10: {hits_10:.3f}")
    print()

In [0]:
# ============================================================
# TASK 15: TRANSFER LEARNING FOR GRAPH TASKS
# ============================================================
# Use pre-trained node embeddings as initialization or features
# for downstream graph neural networks or other models.
#
# Transfer Learning Pipeline:
#   1. Pre-train: Learn general embeddings on large graph
#      Phi_pretrained = DeepWalk/Node2Vec(G_large)
#   2. Fine-tune: Adapt embeddings for specific task
#      Phi_finetuned = FineTune(Phi_pretrained, task_labels)
#
# Benefits:
#   - Better initialization than random
#   - Faster convergence
#   - Better generalization with limited labels
#   - Captures structural prior knowledge
#
# Mathematical connection:
#   Random init: Phi_0 ~ N(0, sigma^2 I)
#   Pre-trained: Phi_0 = Phi_pretrained (encodes graph structure)
#   Fine-tuning: Phi* = argmin L_task(Phi) + lambda||Phi - Phi_0||^2
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim

print("=" * 60)
print("TASK 15: TRANSFER LEARNING FOR GRAPH TASKS")
print("=" * 60)

# Simple neural classifier
class SimpleClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, output_dim)
        )
    def forward(self, x):
        return self.net(x)

# Prepare data
nodes_sorted = sorted(G.nodes())
labels = np.array([community_map[n] for n in nodes_sorted])
_, pretrained_embs = n2v_homophily.get_all_embeddings()

# Train/test split
train_idx, test_idx = train_test_split(
    range(len(nodes_sorted)), test_size=0.3, random_state=42, stratify=labels
)

train_indices = torch.LongTensor(train_idx)
test_indices = torch.LongTensor(test_idx)
train_labels = torch.LongTensor(labels[train_idx])
test_labels = torch.LongTensor(labels[test_idx])

# Random vs Pre-trained features
torch.manual_seed(42)
random_embs = torch.randn(len(nodes_sorted), 64)
pretrained_tensor = torch.FloatTensor(pretrained_embs)

criterion = nn.CrossEntropyLoss()

# Train with random embeddings
print("\nTraining with random embeddings...")
model_rand = SimpleClassifier(64, 32, 2)
opt_r = optim.Adam(model_rand.parameters(), lr=0.01)
losses_random = []
for epoch in range(200):
    model_rand.train(); opt_r.zero_grad()
    loss = criterion(model_rand(random_embs[train_indices]), train_labels)
    loss.backward(); opt_r.step()
    losses_random.append(loss.item())

# Train with pre-trained embeddings
print("Training with pre-trained Node2Vec embeddings...")
model_pre = SimpleClassifier(64, 32, 2)
opt_p = optim.Adam(model_pre.parameters(), lr=0.01)
losses_pretrained = []
for epoch in range(200):
    model_pre.train(); opt_p.zero_grad()
    loss = criterion(model_pre(pretrained_tensor[train_indices]), train_labels)
    loss.backward(); opt_p.step()
    losses_pretrained.append(loss.item())

# Evaluate
model_rand.eval(); model_pre.eval()
with torch.no_grad():
    acc_rand = (model_rand(random_embs[test_indices]).argmax(1) == test_labels).float().mean().item()
    acc_pre = (model_pre(pretrained_tensor[test_indices]).argmax(1) == test_labels).float().mean().item()

print(f"\nResults after 200 epochs:")
print(f"  Random embeddings:     Accuracy = {acc_rand:.4f}")
print(f"  Pre-trained (Node2Vec): Accuracy = {acc_pre:.4f}")
print(f"  Improvement: +{(acc_pre - acc_rand)*100:.1f} percentage points")

# Label efficiency study
print("\nLabel Efficiency Study:")
print("(Accuracy with different fractions of training labels)\n")

fractions = [0.1, 0.2, 0.3, 0.5, 0.7, 1.0]
acc_random_list, acc_pretrained_list = [], []

for frac in fractions:
    n_train = max(2, int(len(train_indices) * frac))
    subset_idx = train_indices[:n_train]
    subset_labels = train_labels[:n_train]
    
    # Random
    m_r = SimpleClassifier(64, 32, 2)
    opt = optim.Adam(m_r.parameters(), lr=0.01)
    for _ in range(200):
        m_r.train(); opt.zero_grad()
        criterion(m_r(random_embs[subset_idx]), subset_labels).backward(); opt.step()
    m_r.eval()
    with torch.no_grad():
        acc_r = (m_r(random_embs[test_indices]).argmax(1) == test_labels).float().mean().item()
    acc_random_list.append(acc_r)
    
    # Pre-trained
    m_p = SimpleClassifier(64, 32, 2)
    opt = optim.Adam(m_p.parameters(), lr=0.01)
    for _ in range(200):
        m_p.train(); opt.zero_grad()
        criterion(m_p(pretrained_tensor[subset_idx]), subset_labels).backward(); opt.step()
    m_p.eval()
    with torch.no_grad():
        acc_p = (m_p(pretrained_tensor[test_indices]).argmax(1) == test_labels).float().mean().item()
    acc_pretrained_list.append(acc_p)
    
    print(f"  {frac*100:>5.0f}% labels ({n_train:>2} samples): "
          f"Random={acc_r:.3f}, Pre-trained={acc_p:.3f}, Delta={acc_p-acc_r:+.3f}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(losses_random, label='Random Init', alpha=0.7, color='red')
axes[0].plot(losses_pretrained, label='Pre-trained (Node2Vec)', alpha=0.7, color='blue')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Training Loss')
axes[0].set_title('Learning Curves: Random vs Pre-trained')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(fractions, acc_random_list, 'ro-', label='Random Init', markersize=8)
axes[1].plot(fractions, acc_pretrained_list, 'bs-', label='Pre-trained (Node2Vec)', markersize=8)
axes[1].set_xlabel('Fraction of Training Labels'); axes[1].set_ylabel('Test Accuracy')
axes[1].set_title('Label Efficiency: Pre-trained vs Random')
axes[1].legend(); axes[1].grid(True, alpha=0.3); axes[1].set_ylim(0, 1.1)

plt.suptitle('Transfer Learning Benefits', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nKey Takeaway: Pre-trained embeddings converge faster AND need fewer labels!")

---
# Summary and Key Takeaways

## DeepWalk vs Node2Vec — When to Use What?

| Criterion | DeepWalk | Node2Vec |
|-----------|----------|----------|
| Simplicity | Simpler, fewer hyperparameters | More complex, requires tuning p, q |
| Homophily (communities) | Good | Excellent (low q) |
| Structural equivalence | Limited | Good (high q) |
| Computation | Faster (no precomputation) | Slower (precompute transition probs) |
| Use when... | Quick baseline, homogeneous graphs | Need fine-grained control, heterogeneous structure |

## Downstream Tasks Summary

| # | Task | Key Technique | Metric |
|---|------|--------------|--------|
| 1 | Node Classification | Logistic Regression / GBM on embeddings | Accuracy, F1 |
| 2 | Link Prediction | Hadamard product + classifier | AUC, AP |
| 3 | Community Detection | K-Means / HDBSCAN on embeddings | NMI, ARI |
| 4 | Recommendation | Cosine similarity in embedding space | Hit Rate |
| 5 | Similarity Search | FAISS nearest neighbor | Recall@K |
| 6 | Visualization | PCA / t-SNE / UMAP projection | Visual separation |
| 7 | Feature Engineering | Embeddings + structural features | CV Accuracy |
| 8 | Anomaly Detection | Isolation Forest / LOF / Deviation | Anomaly Score |
| 9 | Fraud Ring Discovery | HDBSCAN + risk scoring | Precision, Recall |
| 10 | Search & Ranking | Vector similarity ranking | MRR, NDCG |
| 11 | Cold-Start Recs | Neighbor mean approximation | Community Hit Rate |
| 12 | Entity Resolution | Threshold-based pair matching | Precision |
| 13 | Customer Segmentation | Hierarchical clustering on combined features | NMI, Silhouette |
| 14 | KG Completion | TransE-style scoring | Hits@K |
| 15 | Transfer Learning | Pre-trained init for neural nets | Label Efficiency |

## The Big Picture

$$\boxed{\text{Graph} \xrightarrow{\text{Random Walks}} \text{Sequences} \xrightarrow{\text{Word2Vec}} \text{Embeddings} \xrightarrow{\text{Any ML}} \text{Predictions}}$$

DeepWalk and Node2Vec are **representation learning** algorithms. Their primary purpose is to transform a graph into dense vector representations, after which **almost any standard machine learning algorithm** can operate on those vectors instead of the original graph.